In [29]:
#Terve tuvastusskript kaardilehe nimede jaoks, näidiseks leht 64593 (Virumaa).
#Kõigepealt tehakse vajalik eeltöötlus, siis tuvastus erinevate lähenemistega ja lõpuks viiakse nimed kokku.
#Selles skriptis vajalikud failid on kõvakettale alla laaditud.

In [ ]:
#Sisukord.

#1. Kehtivate põhikaartide eeltöötlus.

##1a. Koordinaatsüsteemi kontroll.
##1b. Kehtiva EPK ümberprojitseerimine.
##1c. Testialale vastava vektorkihi loomine ja selle puhverdamine (200 m).
##1d. Kehtiva EPK mosaiigi loomine.

#2. 20T kaardi eeltöötlus.

##2a. 20-tuhandise EPK koordinaatsüsteemi paikapanek.
##2b. Raami lahtilõikamine.
##2b.1 Alternatiivne töökäik.
##2c. EPK 20T rastrite Lõikamine.
##2d. Rastrite lõikamine testala pindobjektiga.
##2e. Kokkusulandamine ehk 20T EPK mosaiigi loomine.
##2f. Saadud mosaiigi binariseerimine.

#3. Vanema digitaalkaardi (EPK 10T) eeltöötlus.

##3a. Koordinaatsüsteemi kontroll.
##3b. Koordinaatsüsteemi lisamine metaandmetesse.
##3c. Vajalike kaustade defineerimine.
##3d. Rastrite lõikamine testala pindobjektiga.
##3e. Kokkusulandamine ehk 10T EPK mosaiigi loomine.
##3f. Värviskaala kontroll.
##3g. Tulemuse binariseerimine.

#4. EasyOCR, näidisena EPK 10T mosaiik.

##4a. Mosaiigi lugemine.
##4b. Tuvastus.
##4c. Vähimad ümbritsevad ristkülikud (VÜR, bbox) kirjade ümber, väike fail ja hästi nähtavad kirjad.
##4d. Muutmine GeoJSONiks ja puhtarvuliste tulemuste eemaldamine.
##4d.1. Ilma regulaaravaldiseta (selle saab nt GIS-is teha).
##4d.2. Regulaarvaldisega.
##4e. OCR-tulemuste kirjutamine punktidena faili.
##4e.1 Näidis tulemuste sulandamisest testandmete abil.

#5. Mitmemodaalse mudeliga tuvastus trükikaardi ja Gemini 3 näitel.

##5a. Ettevalmistused: ruutudeks jagamine (tiling).
##5b. Keelemudeli tuvastusskript, Gemini 3.
##5c. Tulemuste puhastamine: formaat korda ja jäetakse ainult koordinaatidega tulemused. Ka arvuliste tulemuste eemaldamine.
##5d. Pikslikoordinaaditide muutmine ristkoordinaatideks ja tuvastatud punktide faili loomine.
#5d.1. Näide, kuidas korduvaid punkte on testandmete abil eemaldatud.
##5e. Andmete klasterdumise lahendamine teise lehe näitel.

#6. Kombineeritud meetod - keelemudel tuvastab nimed, mis jäävad EasyOCR-i ristkülikute sisse.
##6a. Nimede väljalõikamine bboxide abil.
##6b. Võre loomine.
##6c. Võre alusel keelemudeli sisendpiltide loomine.
##6d. Tuvastusskript.
##6e. Kombineeritud meetodi tulemuste puhastamine ja tulemusfaili loomine.
##6f. Nimede kokkuliitmine.

#7. Ajaliste kihistute kokkuviimine.
##7a. Ettevalmistus.
##7b. Kombineeritud meetodi alusel saadud erinevate aegade tulemuste kokkuviiimine (referentsiks praegused nimed)
##7b.1 Näidis mitme kihiga. Praegused nimed ja kehtiv põhikaart.
##7b.2 Näidis eelmise ajalise kihiga.
#7c. Teisenduskauguse alusel kahe tuvastustulemuse kokkuviimine. Näidiseks 0-kaugus, et saada need, mis pole muutunud.

In [4]:
#Vajalikud impordid.
import cv2
import torch
import easyocr
import numpy as np
import geopandas as gpd
import json
import math
import os
from shapely.geometry import box
from shapely.geometry import mapping
from shapely.geometry import Point
import pandas as pd
from pathlib import Path
import rasterio
from rasterio.crs import CRS
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.merge import merge
from rasterio.mask import mask
from rasterio.windows import from_bounds
from rasterio.windows import Window
from rasterio.merge import merge
from rasterio.shutil import copy
from rasterio.transform import rowcol
import re
from pyproj import CRS
from Levenshtein import distance

In [36]:
#1. Kehtivate põhikaartide eeltöötlus.
#1a. Koordinaatsüsteemi kontroll.
#Katsetatud aasta 2026 mustvalged EPK-failid on olnud Eesti riiklikust koordinaatsüsteemist erinevad, mispärast soovitatakse kontrolli.
#Eeldame, et ümbritsevatel on sama koordinaatsüsteem ja sellepärast testime ühe lehega.
#Aasta 2025 failidega seda probleemi ei peaks olema, ent tasub siiski üle kontrollida.

# Rasterfaili tee
raster_path_epk2026 = Path("64593_epk_mv_2026/64593.tif") #Asenda oma failiteega.

# Ava raster ja loe CRS
with rasterio.open(raster_path_epk2026) as src:
    crs = src.crs

    #Prindi CRS
    print("CRS:", crs)

    #Prindi EPSG-kood
    if crs:
        epsg_code = crs.to_epsg()  #Tagastab EPSG-koodi (int)
        print("EPSG kood:", epsg_code)
    else:
        print("CRS pole määratud!")

CRS: COMPD_CS["Estonian Coordinate System of 1997 + Baltic 1977 height",PROJCS["Estonian Coordinate System of 1997",GEOGCS["EST97",DATUM["Estonia_1997",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6180"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4180"]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",57.5175539305556],PARAMETER["central_meridian",24],PARAMETER["standard_parallel_1",59.3333333333333],PARAMETER["standard_parallel_2",58],PARAMETER["false_easting",500000],PARAMETER["false_northing",6375000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Northing",NORTH],AXIS["Easting",EAST],AUTHORITY["EPSG","3301"]],VERT_CS["Baltic 1977 height",VERT_DATUM["Baltic 1977",2005,AUTHORITY["EPSG","5105"]],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Gravity-related height",UP],AUTHORITY["EPSG","5705"]]]
EPSG kood: None


In [37]:
#1b. Kehtiva EPK ümberprojitseerimine.
#Vältimaks hilisemat mäluprobleemi (memory error) mosaiigi tekitamisel käsitletakse seda lehte, mis on täielikult testiala sees, eraldi.
#See tasub siis pärast rastrite lõikamist tõsta samasse lõigatud kausta (koht on skripti märgitud).

dst_crs = 'EPSG:3301'

with rasterio.open("64593_epk_mv_2026/64593.tif") as src:
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds)
    kwargs = src.meta.copy()
    kwargs.update({
        'crs': dst_crs,
        'transform': transform,
        'width': width,
        'height': height,
        'nodata': src.nodata,
        'dtype': src.dtypes[0],
        'compress':'lzw',       
        'tiled':True,             
        'bigtiff':"if_safer"       
    })

    with rasterio.open("64593_epk_mv_2026/64593_3301.tif", 'w', **kwargs) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest)

print("Valmis.")

Valmis.


In [38]:
#Kontroll.
print(dst.crs)

EPSG:3301


In [40]:
#Vajalike kaustade defineerimine.

original_folder = Path("64593_epk_mv_2026/64593_ymbrus")
#Kaust, milles asuvad praeguse mustvalge EPK lehed ilma selleta, mis on testiala poolt täielikult kaetud.
reprojected_folder = Path("64593_epk_mv_2026/64593_ymbrus/64593_ymbrus_3301")
#Sellesse kausta pannakse ümberprojitseeritud lehed.

os.makedirs(reprojected_folder, exist_ok=True) #Kaust luuakse siin.

In [41]:
#Koordinaatsüsteem korda ülejäänutes kausta failides.
dst_crs = "EPSG:3301"

for filename in os.listdir(original_folder):
    if not filename.lower().endswith(".tif"):
        continue

    input_path = os.path.join(original_folder, filename)
    output_path = os.path.join(reprojected_folder, filename.replace(".tif", "_3301.tif"))

    print("Ümberprojitseeritakse faili:", filename)

    with rasterio.open(input_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds
        )

        kwargs = src.meta.copy()
        kwargs.update({
            'crs': dst_crs,
            'transform': transform,
            'width': width,
            'height': height,
            'nodata': src.nodata,
            'compress':'lzw',         
            'tiled':True,          
            'bigtiff':"if_safer"       
        })

        with rasterio.open(output_path, "w", **kwargs) as dst_folder:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst_folder, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.nearest
                )

print("Valmis.")

Ümberprojitseeritakse faili: 64582.tif
Ümberprojitseeritakse faili: 64584.tif
Ümberprojitseeritakse faili: 64591.tif
Ümberprojitseeritakse faili: 64592.tif
Ümberprojitseeritakse faili: 64594.tif
Ümberprojitseeritakse faili: 64682.tif
Ümberprojitseeritakse faili: 64691.tif
Ümberprojitseeritakse faili: 64692.tif
Valmis.


In [39]:
#1c. Testialale vastava vektorkihi loomine ja selle puhverdamine (200 m).

raster_path_testiala = "64593_epk_mv_2026/64593_3301.tif"

with rasterio.open(raster_path_testiala) as src:
    bounds = src.bounds
    crs = src.crs

extent_polygon = box(
    bounds.left,
    bounds.bottom,
    bounds.right,
    bounds.top
)

gdf = gpd.GeoDataFrame(
    {"nimi": ["64593_extent"]},
    geometry=[extent_polygon],
        crs=crs
)

gdf_64593_buffered = gdf.copy()
gdf_64593_buffered["geometry"] = gdf.geometry.buffer(
    200,
    join_style=2   # 1=ümar (round), 2=terav (mitre), 3=viltune (bevel)
)

gdf_64593_buffered.to_file("64593_epk_mv_2026/64593_buffer200m.shp")

print("Valmis.")

Valmis.


In [42]:
#1d. Kehtiva EPK mosaiigi loomine.
#Vajalikud kaustad.

clipped_folder = Path("64593_epk_mv_2026/64593_ymbrus/64593_ymbrus_clipped")
#Sellesse kausta tulevad testialaga lõigatud rastri tükid.
buffer_shp = Path("64593_epk_mv_2026/64593_buffer200m.shp")
mosaic_output = Path("64593_epk_mv_2026/64593_ymbrus/64593_mosaic_3301.tif")

os.makedirs(clipped_folder, exist_ok=True)

In [44]:
#Tsükliga käiakse üle kõik rastrid ja lõigatakse puhverdatud pinnaga.

buffer_gdf = gpd.read_file(buffer_shp)

geoms = [mapping(geom) for geom in buffer_gdf.geometry]

for file in os.listdir(reprojected_folder):
    if not file.lower().endswith(".tif"):
        continue

    raster_path = os.path.join(reprojected_folder, file)
    out_path = os.path.join(clipped_folder, f"clip_{file}")

    with rasterio.open(raster_path) as src:
        # CRS safety check
        if src.crs != buffer_gdf.crs:
            raise ValueError(f"CRS mismatch failis {file}")

        clipped, clipped_transform = mask(
            src,
            geoms,
            crop=True,
            nodata=src.nodata
        )
    #Kui tekib "CRS mismatch", asendada eelmine järgmisega:
    #with rasterio.open(raster_path) as src:

        #buffer_local = buffer_gdf.to_crs(src.crs)

        #clipped, clipped_transform = mask(
            #src,
            #buffer_local.geometry,
            #crop=True,
            #nodata=src.nodata
        #)

        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "height": clipped.shape[1],
            "width": clipped.shape[2],
            "transform": clipped_transform,
            "compress":"lzw",          
            "tiled":True,              
            "bigtiff":"if_safer"      
        })

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(clipped)

    print(f"Lõigatud: {file}")
print("Valmis.")

Lõigatud: 64582_3301.tif
Lõigatud: 64584_3301.tif
Lõigatud: 64591_3301.tif
Lõigatud: 64592_3301.tif
Lõigatud: 64594_3301.tif
Lõigatud: 64682_3301.tif
Lõigatud: 64691_3301.tif
Lõigatud: 64692_3301.tif


In [46]:
#NB!
#Selle koha peal tõsta/kopeeri oma arvutis täielikult testialaga kaetud kaardileht sellesse kausta, 
#mis sai defineeritud kui "clipped_folder".
#Antud juhul on selleks 64593_3301.tif.
#Kui järgmine annab veateateks "unable to allocate x bytes...", proovi sulgeda teisi lahtiseid programme.

In [50]:
#Lõigatud rastrite kokkupanek mosaiigiks.

src_files = []

for file in os.listdir(clipped_folder):
    if file.lower().endswith(".tif"):
        src = rasterio.open(os.path.join(clipped_folder, file))
        src_files.append(src)

mosaic, out_transform = merge(src_files)

out_meta = src_files[0].meta.copy()

out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform,
    "dtype": "float32",
    "compress": "lzw",   
    "predictor": 2, #Kompresseerimise jaoks.
    "tiled": True,
    "blockxsize": 256,
    "blockysize": 256,
    "BIGTIFF": "YES"
})

with rasterio.open(mosaic_output, "w", **out_meta) as dest:
    dest.write(mosaic)

for src in src_files:
    src.close()

print("Valmis.")

In [51]:
#Kontrolli binaarsust.
with rasterio.open(mosaic_output) as src:
    arr = src.read(1)

print(np.unique(arr))

[0. 1.]


In [ ]:
#2. 20T kaardi eeltöötlus.
#2a. 20-tuhandise EPK koordinaatsüsteemi paikapanek.
#Erinevatel lehtedel on erinevad koordinaatsüsteemid küljes.
#Testida ühe lehega ja tegutseda vastavalt tulemusele.

In [59]:
#Testimine ühe lehega.

test = Path("64593_epk_mv_2026/64593_EPK_20T/6458_2023_pohikaart_20T.tif")

# Ava raster ja loe CRS
with rasterio.open(test) as src:
    crs = src.crs

    #Prindi CRS
    print("CRS:", crs)

    #Prindi EPSG-kood
    if crs:
        epsg_code = crs.to_epsg()  #Tagastab EPSG-koodi (int)
        print("EPSG kood:", epsg_code)
    else:
        print("CRS pole määratud!")

CRS: LOCAL_CS["Estonian Coordinate System of 1997",UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
EPSG kood: None


In [ ]:
#Kui prinditakse "CRS: None CRS pole määratud!", valida alternatiiv A.
#Kui prinditakse "CRS: LOCAL_CS["Estonian Coordinate System of 1997",UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],
#AXIS["Northing",NORTH]] EPSG kood: None", valida alternatiiv B.

In [ ]:
#Alternatiiv A.

epk_20T_folder = Path("64593_epk_mv_2026/64593_EPK_20T")
epk_20T_reprojected_folder = Path("64593_epk_mv_2026/64593_EPK_20T_3301")

os.makedirs(epk_20T_reprojected_folder, exist_ok=True)

dst_crs = "EPSG:3301"

for filename in os.listdir(epk_20T_folder):
    if not filename.lower().endswith(".tif"):
        continue

    input_path = os.path.join(epk_20T_folder, filename)
    output_path = os.path.join(epk_20T_reprojected_folder, filename.replace(".tif", "_3301.tif"))

    print("Ümberprojitseeritakse faili:", filename)

    with rasterio.open(input_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds
        )

        kwargs = src.meta.copy()
        kwargs.update({
            'crs': dst_crs,
            'transform': transform,
            'width': width,
            'height': height,
            'nodata': src.nodata,
            'compress':'lzw',          
            'tiled':True,             
            'bigtiff':"if_safer"    
        })

        with rasterio.open(output_path, "w", **kwargs) as dst_folder:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst_folder, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.nearest
                )

print("Valmis.")

In [63]:
#Alternatiiv B.

from rasterio.crs import CRS

epk_20T_folder = Path("64593_epk_mv_2026/64593_EPK_20T")
#Selles kaustas asuvad kaardifailid.
epk_20T_reprojected_folder = Path("64593_epk_mv_2026/64593_EPK_20T_3301")

os.makedirs(epk_20T_reprojected_folder, exist_ok=True)

target_crs = CRS.from_epsg(3301)

for file in os.listdir(epk_20T_folder):
    if not file.lower().endswith(".tif"):
        continue

    in_path = epk_20T_folder / file
    out_name = file.replace(".tif", "_3301.tif")
    out_path = epk_20T_reprojected_folder / out_name

    with rasterio.open(in_path) as src:
        need_fix = (
            src.crs is None
            or src.crs.to_epsg() is None
        )

        meta = src.meta.copy()

        meta.update({
            "compress": "lzw",
            "tiled": True,
            "bigtiff": "if_safer"
        })

        if need_fix:
            print("Parandatakse faili:", file)
            meta.update({"crs": target_crs})
        else:
            print("Kopeeritakse (CRS juba korras) fail:", file)

        with rasterio.open(out_path, "w", **meta) as dst:
            for i in range(1, src.count + 1):
                dst.write(src.read(i), i)

print("Valmis.")

Parandatakse faili: 6458_2023_pohikaart_20T.tif
Kopeeritakse (CRS juba korras) fail: 6459_2016_pohikaart_20T.tif
Kopeeritakse (CRS juba korras) fail: 6469_2016_pohikaart_20T.tif
Valmis.


In [66]:
#2b. Raami lahtilõikamine.
#Leitakse kaardiala ja luuakse sellele vastav pindobjekt.
#Kaardiala on 10x10 km.
#Kui see ei anna piisavalt häid tulemusi (pindobjekt ei katu rastriga), proovida skripti kohas 2b.1 või tõsta GIS-is pind õige koha peale.

input_folder = epk_20T_reprojected_folder
output_polygons = Path("64593_epk_mv_2026/64593_EPK_20T_3301/clipped_to_area")

os.makedirs(output_polygons, exist_ok=True)

def detect_map_inside_frame_tight(
    raster_path,
    frame_width_m=10000,
    frame_height_m=10000,
    out_polygon_path=None,
    tolerance=0.04, #Kui pindobjekte ei leita, proovida muuta seda väärtust aga võta väikseim töötav. Vahemikus 0,04 - 0,22 on seni õnnestunud.
    scale=4         
):
    with rasterio.open(raster_path) as src:

        #resolutsioon
        res_x = abs(src.transform.a)
        res_y = abs(src.transform.e)

        #oodatav ala suurus pikslites
        expected_w_px_full = frame_width_m / res_x
        expected_h_px_full = frame_height_m / res_y

        #downsamplingu jaoks kohandamine
        expected_w_px = expected_w_px_full / scale
        expected_h_px = expected_h_px_full / scale

        #suuruse downsampling
        out_h = src.height // scale
        out_w = src.width // scale

        #loe tekkinud pilti
        if src.count >= 3:
            r = src.read(1, out_shape=(out_h, out_w)).astype(np.float32)
            g = src.read(2, out_shape=(out_h, out_w)).astype(np.float32)
            b = src.read(3, out_shape=(out_h, out_w)).astype(np.float32)
            img = (0.299*r + 0.587*g + 0.114*b).astype(np.uint8)
        else:
            img = src.read(1, out_shape=(out_h, out_w)).astype(np.uint8)

        transform = src.transform
        crs = src.crs

    #raami leidmine
    edges = cv2.Canny(img, 50, 150)

    contours, _ = cv2.findContours(
        edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    print(f"Kontuure leitud: {len(contours)}")

    img_h, img_w = img.shape
    best_rect = None
    best_score = float("inf")

    #leia parim tingimustele vastav ristkülik
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)

        # skip tiny junk
        if w < img_w * 0.3 or h < img_h * 0.3:
            continue

        #filtreerimine
        if not (expected_w_px*(1-tolerance) <= w <= expected_w_px*(1+tolerance)):
            continue
        if not (expected_h_px*(1-tolerance) <= h <= expected_h_px*(1+tolerance)):
            continue

        size_score = abs(w - expected_w_px) + abs(h - expected_h_px)

        if size_score < best_score:
            best_score = size_score
            best_rect = (x, y, w, h)

    if best_rect is None:
        raise RuntimeError("Kaardiraami ei leitud.")

    x, y, w, h = best_rect

    #tagasi pikslikoordinaatideks
    x_full = x * scale
    y_full = y * scale
    w_full = w * scale
    h_full = h * scale

    #1. leia raami kese
    cx_px = x_full + w_full / 2
    cy_px = y_full + h_full / 2

    #pikslikoordinaatideks ristkoordinaatideks
    cx, cy = rasterio.transform.xy(transform, cy_px, cx_px)

    #2. 10x10 km kast
    half_w = frame_width_m / 2
    half_h = frame_height_m / 2

    crop_bounds = (
        cx - half_w,
        cy - half_h,
        cx + half_w,
        cy + half_h,
    )

    geom = box(*crop_bounds)
    gdf = gpd.GeoDataFrame(geometry=[geom], crs=crs)

    if out_polygon_path:
        gdf.to_file(out_polygon_path)
        print("Salvestatud pindobjekt:", out_polygon_path)

    return gdf

def epk_batch_detect_tight(input_folder, output_folder):
    input_folder = Path(input_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(exist_ok=True)

    tif_files = list(input_folder.glob("*.tif"))

    print(f"Leitud {len(tif_files)} rasterfaili.")

    for tif in tif_files:
        print(f"\nTöödeldakse faili: {tif.name}")

        try:
            out_poly = output_folder / f"{tif.stem}_frame.shp"

            detect_map_inside_frame_tight(
                tif,
                frame_width_m=10000,
                frame_height_m=10000,
                out_polygon_path=out_poly
            )

        except Exception as e:
            print(f"VIGA: {tif.name}")
            print("Põhjus:", e)

epk_batch_detect_tight(
    input_folder,
    output_polygons
)

Leitud 3 rasterfaili.

Töödeldakse faili: 6458_2023_pohikaart_20T_3301.tif
Kontuure leitud: 37763
Salvestatud pindobjekt: C:\Users\antti\Documents\Magister\Makatöö\Uudet\64593_epk_mv_2026\64593_EPK_20T_3301\clipped_to_area\6458_2023_pohikaart_20T_3301_frame.shp

Töödeldakse faili: 6459_2016_pohikaart_20T_3301.tif
Kontuure leitud: 29969
Salvestatud pindobjekt: C:\Users\antti\Documents\Magister\Makatöö\Uudet\64593_epk_mv_2026\64593_EPK_20T_3301\clipped_to_area\6459_2016_pohikaart_20T_3301_frame.shp

Töödeldakse faili: 6469_2016_pohikaart_20T_3301.tif
Kontuure leitud: 29389
Salvestatud pindobjekt: C:\Users\antti\Documents\Magister\Makatöö\Uudet\64593_epk_mv_2026\64593_EPK_20T_3301\clipped_to_area\6469_2016_pohikaart_20T_3301_frame.shp


In [ ]:
#2b.1 Alternatiiv eelmisele, kui see annab liiga suuri vahesid.
#Leitakse kaardiala ja luuakse sellele vastav pindobjekt.
#Kaardiala on 10x10 km.

input_folder = epk_20T_reprojected_folder
output_polygons = Path("64593_epk_mv_2026/64593_EPK_20T_3301/clipped_to_area")

os.makedirs(output_polygons, exist_ok=True)

def detect_map_inside_frame_tight(
    raster_path,
    frame_width_m=10000,
    frame_height_m=10000,
    out_polygon_path=None,
    tolerance=0.12,
    scale=4
):

    with rasterio.open(raster_path) as src:

        res_x = abs(src.transform.a)
        res_y = abs(src.transform.e)

        expected_w_px_full = frame_width_m / res_x
        expected_h_px_full = frame_height_m / res_y

        expected_w_px = expected_w_px_full / scale
        expected_h_px = expected_h_px_full / scale

        out_h = src.height // scale
        out_w = src.width // scale

        if src.count >= 3:
            r = src.read(1, out_shape=(out_h, out_w)).astype(np.float32)
            g = src.read(2, out_shape=(out_h, out_w)).astype(np.float32)
            b = src.read(3, out_shape=(out_h, out_w)).astype(np.float32)
            img = (0.299*r + 0.587*g + 0.114*b).astype(np.uint8)
        else:
            img = src.read(1, out_shape=(out_h, out_w)).astype(np.uint8)

        transform = src.transform
        crs = src.crs

    # servade (edges) leidmine
    edges = cv2.Canny(img, 50, 150)

    kernel = np.ones((5,5), np.uint8)
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(
        edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    print(f"Kontuure leitud: {len(contours)}")

    img_h, img_w = img.shape

    best_rect = None
    best_cnt = None
    best_score = float("inf")

    #Leida optimaalne ristkülik.
    for cnt in contours:

        x, y, w, h = cv2.boundingRect(cnt)

        if w < img_w * 0.3 or h < img_h * 0.3:
            continue

        if not (expected_w_px*(1-tolerance) <= w <= expected_w_px*(1+tolerance)):
            continue

        if not (expected_h_px*(1-tolerance) <= h <= expected_h_px*(1+tolerance)):
            continue

        size_score = abs(w - expected_w_px) + abs(h - expected_h_px)

        if size_score < best_score:
            best_score = size_score
            best_rect = (x, y, w, h)
            best_cnt = cnt

    if best_rect is None:
        raise RuntimeError("Kaardiraami ei leitud.")

    x, y, w, h = best_rect

    #Valikuline: veaotsing. Kui pindobjekti ja kaardiala vahel on ikka lõhe, uurida siintoodud väärtusi (väljakommenteeritud).
    M = cv2.moments(best_cnt)

    bbox_cx = x + w/2
    bbox_cy = y + h/2

    centroid_cx = M["m10"] / M["m00"]
    centroid_cy = M["m01"] / M["m00"]

    #print("Bounding center px:", bbox_cx, bbox_cy)
    #print("Centroid px:", centroid_cx, centroid_cy)

    #Resolutsioon.
    cx_px = centroid_cx * scale
    cy_px = centroid_cy * scale

    cx, cy = rasterio.transform.xy(
        transform,
        cy_px,
        cx_px,
        offset="center"
    )

    #Luua alale vastav pindobjekt.
    half_w = frame_width_m / 2
    half_h = frame_height_m / 2

    crop_bounds = (
        cx - half_w,
        cy - half_h,
        cx + half_w,
        cy + half_h,
    )

    geom = box(*crop_bounds)

    gdf = gpd.GeoDataFrame(
        geometry=[geom],
        crs=crs
    )

    if out_polygon_path:
        gdf.to_file(out_polygon_path)
        print("Salvestatud pindobjekt:", out_polygon_path)

    return gdf


def epk_batch_detect_tight(input_folder, output_folder):

    input_folder = Path(input_folder)
    output_folder = Path(output_folder)

    output_folder.mkdir(exist_ok=True)

    tif_files = list(input_folder.glob("*.tif"))

    print(f"Leitud {len(tif_files)} rasterfaili.")

    for tif in tif_files:

        print(f"\nTöödeldakse faili: {tif.name}")

        try:

            out_poly = output_folder / f"{tif.stem}_frame_new.shp"

            detect_map_inside_frame_tight(
                tif,
                frame_width_m=10000,
                frame_height_m=10000,
                out_polygon_path=out_poly
            )

        except Exception as e:

            print(f"VIGA: {tif.name}")
            print("Põhjus:", e)


epk_batch_detect_tight(
    input_folder,
    output_polygons
)

In [67]:
#2c. EPK 20T rastrite Lõikamine.
#Esiti iga lehte saadud alaga.

input_folder = epk_20T_reprojected_folder
area_epk_20T_folder = output_polygons
clipped_epk_20T_folder = Path("64593_epk_mv_2026/64593_EPK_20T_3301/clipped_to_area/map_area")

clipped_epk_20T_folder.mkdir(parents=True, exist_ok=True)

print(f"Leitud {len(list(input_folder.glob('*.tif')))} rasterfaili.")

for raster_path in input_folder.glob("*.tif"):

    print(f"\nTöödeldakse: {raster_path.name}")

    try:
        shp_path = area_epk_20T_folder / f"{raster_path.stem}_frame.shp"
        if not shp_path.exists():
            print("Pindobjekti ei leitud, jäetakse vahele.")
            continue

        gdf = gpd.read_file(shp_path)

        with rasterio.open(raster_path) as src:

            if src.crs != gdf.crs:
                gdf = gdf.to_crs(src.crs)

            #pindobjekti ulatus
            bounds = gdf.total_bounds  # left, bottom, right, top

            #loo rasteraken
            window = from_bounds(*bounds, transform=src.transform)

            #loe loodud aken
            clipped = src.read(window=window)

            clipped_transform = src.window_transform(window)

            meta = src.meta.copy()
            meta.update({
                "height": clipped.shape[1],
                "width": clipped.shape[2],
                "transform": clipped_transform,
                "compress": "lzw",
                "tiled": True,
                "bigtiff": "if_safer"
            })

            out_path = clipped_epk_20T_folder / f"clip_{raster_path.name}"

            with rasterio.open(out_path, "w", **meta) as dst:
                dst.write(clipped)

        print(f"Lõigatud: {raster_path.name}")

    except Exception as e:
        print(f"VIGA: {raster_path.name}")
        print("Põhjus:", e)

print("Valmis.")

Leitud 3 rasterfaili.

Töödeldakse: 6458_2023_pohikaart_20T_3301.tif
Lõigatud: 6458_2023_pohikaart_20T_3301.tif

Töödeldakse: 6459_2016_pohikaart_20T_3301.tif
Lõigatud: 6459_2016_pohikaart_20T_3301.tif

Töödeldakse: 6469_2016_pohikaart_20T_3301.tif
Lõigatud: 6469_2016_pohikaart_20T_3301.tif


In [68]:
#2d. Rastrite lõikamine testala pindobjektiga.

epk_20T_mosaic_folder = Path("64593_epk_mv_2026/64593_EPK_20T_3301/clipped_to_area/map_area/mosaic")
epk_20T_mosaic_folder.mkdir(exist_ok=True)

#ettevalmistus
def get_geoms_for_src(src, buffer_gdf):
    if src.crs != buffer_gdf.crs:
        gdf_proj = buffer_gdf.to_crs(src.crs)
    else:
        gdf_proj = buffer_gdf
    return [mapping(geom) for geom in gdf_proj.geometry]

#Tsükkel.
for file in os.listdir(clipped_epk_20T_folder):
    if not file.lower().endswith(".tif"):
        continue

    raster_path_epk_20T = os.path.join(clipped_epk_20T_folder, file)
    out_path_epk_20T = epk_mosaic_20T_folder / f"clip_{file}"

    with rasterio.open(raster_path_epk_20T) as src:

        geoms = get_geoms_for_src(src, buffer_gdf)

        clipped, clipped_transform = mask(
            src,
            geoms,
            crop=True,
            nodata=src.nodata
        )

        meta = src.meta.copy()
        meta.update({
            "height": clipped.shape[1],
            "width": clipped.shape[2],
            "transform": clipped_transform,
            "compress": "lzw",
            "tiled": True,
            "bigtiff": "if_safer"
        })

        with rasterio.open(out_path_epk_20T, "w", **meta) as dst:
            dst.write(clipped)

    print(f"Lõigatud: {file}")
    
print("Valmis.")

Lõigatud: clip_6458_2023_pohikaart_20T_3301.tif
Lõigatud: clip_6459_2016_pohikaart_20T_3301.tif
Lõigatud: clip_6469_2016_pohikaart_20T_3301.tif
Valmis.


In [70]:
#2e. Kokkusulandamine ehk 20T EPK mosaiigi loomine.

epk_mosaic_20T = Path("64593_epk_mv_2026/64593_EPK_20T_3301/64593_epk_20T_mosaic.tif")

src_files = []

for file in os.listdir(epk_20T_epk_mosaic_folder):
    if file.lower().endswith(".tif"):
        src = rasterio.open(os.path.join(epk_20T_mosaic_folder, file))
        src_files.append(src)

mosaic, out_transform = merge(src_files)

out_meta = src_files[0].meta.copy()

out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform,
    "dtype": "float32",
    "compress": "LZW",  
    "predictor": 2,
    "tiled": True,
    "blockxsize": 256,
    "blockysize": 256,
    "BIGTIFF": "YES"
})

with rasterio.open(epk_mosaic_20T, "w", **out_meta) as dest:
    dest.write(mosaic)

for src in src_files:
    src.close()

print("Valmis.")

Valmis.


In [40]:
#2f. Saadud mosaiigi binariseerimine

in_tif_epk_20T = epk_mosaic_20T
out_tif_epk_20T = Path("64593_epk_mv_2026/64593_EPK_20T_3301/64593_epk_20T_mosaic_bin_new.tif")

#Loe mosaiik
with rasterio.open(in_tif_epk_20T) as src:
    rgb = src.read([1, 2, 3])     # shape: (3, H, W)
    transform = src.transform
    crs = src.crs
    height = src.height
    width = src.width

#pööramine
rgb = np.transpose(rgb, (1, 2, 0))

## RGB -> mustvalge
gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

#normaliseerimine 8-bit-formaati
gray = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

#binariseerimine
binary = cv2.adaptiveThreshold(
    gray,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    blockSize=35, #Kui tulemus on mitterahuldav (nt taustas halle triipe), proovida sättida neid kahte väärtust. Langetada seda ja/või
    C=15 #tõsta seda. Algsed bS=35, C=11.
)

#Kirjutatakse fail.
with rasterio.open(
    out_tif_epk,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="uint8",
    crs=crs,
    transform=transform,
    compress="LZW",
    photometric="MINISBLACK"
) as dst:
    dst.write(binary, 1)

print("Binariseeritud GeoTIFF loodud.")

Binariseeritud GeoTIFF loodud.


In [41]:
#Testime, et on binaarne.

with rasterio.open(r"64593_epk_mv_2026/64593_EPK_20T_3301/64593_epk_20T_mosaic_bin_new.tif") as src:
    arr = src.read(1)

print(np.unique(arr))

[  0 255]


In [42]:
#3. Vanema digitaalkaardi (EPK 10T) eeltöötlus.
#3a. Koordinaatsüsteemi kontroll.

#Testimine ühe lehega.

test10T = Path("64593_epk_mv_2026/64593_EPK_10T/64592_2001_pohikaart_10T.tif")

# Ava raster ja loe CRS
with rasterio.open(test10T) as src:
    crs = src.crs

    #Prindi CRS
    print("CRS:", crs)

    #Prindi EPSG-kood
    if crs:
        epsg_code = crs.to_epsg()  #Tagastab EPSG-koodi (int)
        print("EPSG kood:", epsg_code)
    else:
        print("CRS pole määratud!")

CRS: PROJCS["unnamed",GEOGCS["GRS 1980(IUGG, 1980)",DATUM["unnamed",SPHEROID["unnamed",6378137,298.257222101004]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",57.5175539305556],PARAMETER["central_meridian",24],PARAMETER["standard_parallel_1",59.3333333333333],PARAMETER["standard_parallel_2",58],PARAMETER["false_easting",500000],PARAMETER["false_northing",6375000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
EPSG kood: None


In [ ]:
#3b. Koordinaatsüsteemi lisamine metaandmetesse.

epk_10T_folder = Path("64593_epk_mv_2026/64593_EPK_10T")
epk_10T_reprojected_folder = Path("64593_epk_mv_2026/64593_EPK_10T_3301")

os.makedirs(epk_10T_reprojected_folder, exist_ok=True)

target_crs = CRS.from_epsg(3301)

for file in os.listdir(epk_10T_folder):
    if not file.lower().endswith(".tif"):
        continue

    in_path = epk_10T_folder / file
    out_name = file.replace(".tif", "_3301.tif")
    out_path = epk_10T_reprojected_folder / out_name

    with rasterio.open(in_path) as src:
        need_fix = (
            src.crs is None
            or src.crs.to_epsg() is None
        )

        meta = src.meta.copy()

        meta.update({
            "compress": "lzw",
            "tiled": True,
            "bigtiff": "if_safer"
        })

        if need_fix:
            print("Parandatakse faili:", file)
            meta.update({"crs": target_crs})
        else:
            print("Kopeeritakse (CRS juba korras) fail:", file)

        with rasterio.open(out_path, "w", **meta) as dst:
            for i in range(1, src.count + 1):
                dst.write(src.read(i), i)

print("Valmis.")

In [49]:
#3c. Vajalike kaustade defineerimine.

clipped_epk10T_folder = Path("64593_epk_mv_2026/64593_EPK_10T_3301/64593_EPK10T_clipped")
#Sellesse kausta tulevad testialaga lõigatud rastri tükid.
buffer_shp = Path("64593_epk_mv_2026/64593_buffer200m.shp")
mosaic_output_epk10T = Path("64593_epk_mv_2026/64593_EPK_10T/64593_EPK10T_mosaic_3301.tif")

os.makedirs(clipped_epk10T_folder, exist_ok=True)

print("Valmis.")

Valmis.


In [51]:
#3d. Rastrite lõikamine testala pindobjektiga.
#Ala on 5x5 km.

buffer_gdf = gpd.read_file(buffer_shp)

geoms = buffer_gdf.geometry.values

for file in os.listdir(epk_10T_reprojected_folder):
    if not file.lower().endswith(".tif"):
        continue

    raster_path_10T = os.path.join(epk_10T_reprojected_folder, file)
    out_path_10T = os.path.join(clipped_epk10T_folder, f"clip_{file}")

    with rasterio.open(raster_path_10T) as src:
        # CRS safety check
        if src.crs != buffer_gdf.crs:
            raise ValueError(f"CRS mismatch failis {file}")

        clipped, clipped_transform = mask(
            src,
            geoms,
            crop=True,
            nodata=src.nodata
        )
    #Kui tekib "CRS mismatch", asendada eelmine järgmisega:
    #with rasterio.open(raster_path_10T) as src:

        #buffer_local = buffer_gdf.to_crs(src.crs)

        #clipped, clipped_transform = mask(
            #src,
            #buffer_local.geometry,
            #crop=True,
            #nodata=src.nodata
        #)

        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "height": clipped.shape[1],
            "width": clipped.shape[2],
            "transform": clipped_transform,
            "compress":"lzw",          
            "tiled":True,              
            "bigtiff":"if_safer"      
        })

        with rasterio.open(out_path_10T, "w", **meta) as dst:
            dst.write(clipped)

    print(f"Lõigatud: {file}")
print("Valmis.")

Lõigatud: 64582_2001_pohikaart_10T_3301.tif
Lõigatud: 64584_2001_pohikaart_10T_3301.tif
Lõigatud: 64591_2001_pohikaart_10T_3301.tif
Lõigatud: 64592_2001_pohikaart_10T_3301.tif
Lõigatud: 64593_2001_pohikaart_10T_3301.tif
Lõigatud: 64594_2001_pohikaart_10T_3301.tif
Lõigatud: 64682_2001_pohikaart_10T_3301.tif
Lõigatud: 64691_2001_pohikaart_10T_3301.tif
Lõigatud: 64692_2001_pohikaart_10T_3301.tif


In [53]:
#3e. Kokkusulandamine ehk 10T EPK mosaiigi loomine.

src_files_epk10T = []

for file in os.listdir(clipped_epk10T_folder):
    if file.lower().endswith(".tif"):
        src = rasterio.open(os.path.join(clipped_epk10T_folder, file))
        src_files_epk10T.append(src)

mosaic, out_transform = merge(src_files_epk10T)

out_meta = src_files_epk10T[0].meta.copy()

out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform,
    "dtype": "float32",
    "compress": "lzw",   
    "predictor": 2,
    "tiled": True,
    "blockxsize": 256,
    "blockysize": 256,
    "BIGTIFF": "YES"
})

with rasterio.open(mosaic_output_epk10T, "w", **out_meta) as dest:
    dest.write(mosaic)

for src in src_files_epk10T:
    src.close()

print("Valmis.")

Valmis.


In [55]:
#3f. Värviskaala kontroll.

with rasterio.open(mosaic_output_epk10T) as src:
    arr = src.read(1)

print(np.unique(arr))

#Oodatav tulemus 0-3X ehk mustvalge.

[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19. 20. 21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31. 32. 33. 34.]


In [57]:
#3g. Tulemuse binariseerimine.

in_tif_epk_10T = mosaic_output_epk10T
out_tif_epk_10T = Path("64593_epk_mv_2026/64593_EPK_10T_3301/64593_epk_10T_mosaic_bin.tif")

#Loe mosaiik
with rasterio.open(in_tif_epk_10T) as src:
    gray = src.read(1)
    transform = src.transform
    crs = src.crs
    height = src.height
    width = src.width

# 0–3X väärtused vahemikku 0–255
gray_8bit = (gray.astype(np.float32) * (255.0 / 32)).astype(np.uint8)

#binariseerimine
binary = cv2.adaptiveThreshold(
    gray_8bit,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    blockSize=35, #Kui tulemus on mitterahuldav (nt taustas halle triipe), proovida sättida neid kahte väärtust. Langetada seda ja/või
    C=15 #tõsta seda. Algsed bS=35, C=11.
)

#Kirjutatakse fail.
with rasterio.open(
    out_tif_epk_10T,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="uint8",
    crs=crs,
    transform=transform,
    compress="LZW",
    photometric="MINISBLACK"
) as dst:
    dst.write(binary, 1)

print("Binariseeritud GeoTIFF loodud.")

Binariseeritud GeoTIFF loodud.


In [37]:
#4. EasyOCR, näidisena EPK 10T mosaiik.
#4a. Mosaiigi lugemine.

with rasterio.open(out_tif_epk_10T) as src:
    img = src.read(1)       
    transform = src.transform   # affiinne transformatsioon piksel→kaart
    crs = src.crs

print("Valmis.")

Valmis.


In [38]:
#4b. Tuvastus.

if img.max() <= 1:
    img = (img * 255).astype(np.uint8)

reader = easyocr.Reader(['et'])
results = reader.readtext(img)

#Tulemused on kujul: bbox, tekst, enesekindlus
for (bbox, text, conf) in results:
    print(text, conf, bbox)

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
C:\Users\antti\micromamba\envs\geo312\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Tammetaga 0.9981139347047219 [[np.int32(1023), np.int32(12)], [np.int32(1187), np.int32(12)], [np.int32(1187), np.int32(44)], [np.int32(1023), np.int32(44)]]
220 0.9997368158694606 [[np.int32(5840), 0], [np.int32(5929), 0], [np.int32(5929), np.int32(51)], [np.int32(5840), np.int32(51)]]
216 0.999996902852449 [[np.int32(3700), np.int32(29)], [np.int32(3788), np.int32(29)], [np.int32(3788), np.int32(79)], [np.int32(3700), np.int32(79)]]
218 0.8910921059068677 [[np.int32(4810), np.int32(47)], [np.int32(4899), np.int32(47)], [np.int32(4899), np.int32(103)], [np.int32(4810), np.int32(103)]]
Väikevalja_ 0.9075152237022845 [[np.int32(927), np.int32(129)], [np.int32(1070), np.int32(129)], [np.int32(1070), np.int32(168)], [np.int32(927), np.int32(168)]]
Munamäe 0.9999917095236643 [[np.int32(555), np.int32(262)], [np.int32(696), np.int32(262)], [np.int32(696), np.int32(294)], [np.int32(555), np.int32(294)]]
12 0.9999998314126102 [[np.int32(5155), np.int32(374)], [np.int32(5215), np.int32(374)], 

In [39]:
#4c. Vähimad ümbritsevad ristkülikud (VÜR, bbox) kirjade ümber, väike fail ja hästi nähtavad kirjad.

raster_in = out_tif_epk_10T
raster_out = Path("64593_bbox_mosaic_EPK_10T.tif")

with rasterio.open(raster_in) as src:
    data = src.read(1)        # read grayscale band KAS SEDA ON BINARISEERITUD KAARDIL VAJAGI?!?
    meta = src.meta           #säilitada metaandmed

stroke = 4 #ristküliku raami laius

for (bbox, text, conf) in results:
    coords = [(int(x), int(y)) for [x, y] in bbox]   # [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    x1, y1 = coords[0]      #üleval vasakul
    x2, y2 = coords[2]      #all paremal

    #Joonista ristkülik tuvastustulemuse ümber.
    data[y1:y2, x1:x1+stroke] = 0
    data[y1:y2, x2-stroke:x2] = 0
    data[y1:y1+stroke, x1:x2] = 0
    data[y2-stroke:y2, x1:x2] = 0 
    
meta.update(
    compress="lzw",     
    tiled=True,             
    bigtiff="if_safer"       
)

#Kirjuta fail.
with rasterio.open(raster_out, "w", **meta) as dst:
    dst.write(data, 1)

print("Tulemus salvestatud:", raster_out)

Tulemus salvestatud: C:\Users\antti\Documents\Magister\Makatöö\Uudet\Tulemusfailid\64593_bbox_mosaic_EPK_10T.tif


In [40]:
#4d. Muutmine GeoJSONiks ja puhtarvuliste tulemuste eemaldamine.
##4d.1. Ilma regulaaravaldiseta (selle saab nt GIS-is teha).

raster_path = out_tif_epk_10T
#output_geojson = "64593_bbox_mosaic_EPK_10T.geojson"
output_geojson = "64593_bbox_mosaic_EPK_10T_numbriteta.geojson"

#Selle def ... in text) osaga eemaldatakse arvulised tulemused.
def has_letter(text):
    text = text.strip()
    return any(char.isalpha() for char in text)

with rasterio.open(raster_path) as src:
    transform = src.transform
    crs = src.crs.to_string()

features = []

for (bbox, text, conf) in results:

    if not has_letter(text): #Need kaks rida eemaldavad ka arvulised tulemused.
        continue
    
    geo_coords = []
    
    for (x, y) in bbox:
        #pikslid ristkoordinaatideks
        gx, gy = rasterio.transform.xy(transform, y, x)
        geo_coords.append([gx, gy])

    geo_coords.append(geo_coords[0])  #pindobjekti sulgemine

    features.append({
        "type": "Feature",
        "properties": {"text": text, "confidence": float(conf)},
        "geometry": {"type": "Polygon", "coordinates": [geo_coords]}
    })

geojson = {
    "type": "FeatureCollection",
    "name": "ocr_bounding_boxes",
    "crs": {"type": "name", "properties": {"name": crs}},
    "features": features
}

with open(output_geojson, "w", encoding="utf-8") as f:
    json.dump(geojson, f, indent=2)

print("Salvestatud:", output_geojson, "| CRS:", crs)

Salvestatud: C:/Users/antti/Documents/Magister/Makatöö/Uudet/Tulemusfailid/64593_bbox_mosaic_EPK_10T_numbriteta.geojson | CRS: EPSG:3301


In [ ]:
##4d.2 Regulaarvaldisega.

#Sama sisuga ehk säilitatakse tähed, tühikud keskel ja sidekriipsud.
def clean_text(text):
    text = text.strip()

    #tähemärkide eemaldamine
    text = re.sub(r"[^\w\s\-]", "", text, flags=re.UNICODE)

    #arvude ja alamjoonte eemaldamine
    text = re.sub(r"[_\d]", "", text, flags=re.UNICODE)

    return text.strip()

#Muutmine GeoJSONiks.

raster_path = out_tif_epk_10T
output_geojson = "64593_bbox_mosaic_EPK_10T_numbriteta_regex_cleaned.geojson"

def has_letter(text):
    text = text.strip()
    return any(char.isalpha() for char in text)

with rasterio.open(raster_path) as src:
    transform = src.transform
    crs = src.crs.to_string()

features = []

for (bbox, text, conf) in results:

    #regexiga puhastatud tekst
    text = clean_text(text)

    #mittesobivate vahele jätmine
    if not has_letter(text):
        continue

    geo_coords = []

    for (x, y) in bbox:
        #pikslikoordinaatidest ristkoordinaatideks
        gx, gy = rasterio.transform.xy(transform, y, x)
        geo_coords.append([gx, gy])

    geo_coords.append(geo_coords[0])  #pindobjekti sulgemine

    features.append({
        "type": "Feature",
        "properties": {
            "text": text,
            "confidence": float(conf)
        },
        "geometry": {
            "type": "Polygon",
            "coordinates": [geo_coords]
        }
    })

geojson = {
    "type": "FeatureCollection",
    "name": "ocr_bounding_boxes",
    "crs": {"type": "name", "properties": {"name": crs}},
    "features": features
}

with open(output_geojson, "w", encoding="utf-8") as f:
    json.dump(geojson, f, indent=2)

print("Salvestatud:", output_geojson, "| CRS:", crs)

In [41]:
#4e. OCR-tulemuste kirjutamine punktidena faili.
#Ilma arvuliste tulemusteta.
#Kui tahta need ka kaasata, eemaladada if not has_letter(text): continue.

points = []
texts = []
confs = []

for bbox, text, conf in results:

    #text = clean_text(text)   #See rida võtab regexiga puhastatud tulemused. Kui teha GIS-versioon, võib selle väljakommenteeritud jätta.

    if not has_letter(text): #Need kaks rida eemaldavad ka arvulised tulemused.
        continue
        
    xs = [p[0] for p in bbox]
    ys = [p[1] for p in bbox]
    cx = sum(xs)/4
    cy = sum(ys)/4

    #rasterio järjekord on (row, col) ehk koordinaatide järjekord on siin tähtis (y,x)
    map_x, map_y = rasterio.transform.xy(transform, cy, cx)
    points.append(Point(map_x, map_y))
    texts.append(text)
    confs.append(float(conf))


gdf_texts = gpd.GeoDataFrame({"text": texts, "conf": confs}, geometry=points, crs=crs)
gdf_texts.to_file("64593_detected_mosaic_EPK10T_k6ik_numbriteta.shp")
print("Tulemused kirjutatud faili.")

Tulemused kirjutatud faili.


In [ ]:
#4e.1 Näidis tulemuste sulandamisest testandmete abil.

#Lisatakse confidence (keskmisena).

import geopandas as gpd
import re

bbox_gdf = gpd.read_file(
    "64593_epk_10T_testandmestik.shp"
)

ocr_gdf = gpd.read_file(
    "64593_detected_mosaic_EPK10T_k6ik_numbriteta.shp"
)

merged = []

for idx, bbox_row in bbox_gdf.iterrows():
    bbox_geom = bbox_row.geometry

    inside = ocr_gdf[ocr_gdf.within(bbox_geom)].copy()

    if inside.empty:
        continue

    # sordi vasakult paremale
    inside["x"] = inside.geometry.x
    inside = inside.sort_values("x")

    # merge text
    merged_text = " ".join(inside["text"].astype(str))
    merged_text = re.sub(r"\s+", " ", merged_text).strip()

    #lisa enesekindlus
    mean_conf = inside["conf"].mean()

    center = inside.geometry.union_all().centroid

    merged.append({
        "text": merged_text,
        "conf": mean_conf,
        "geometry": center
    })

merged_gdf = gpd.GeoDataFrame(merged, crs=ocr_gdf.crs)
merged_gdf.to_file("64593_merge_EPK10T_k6ik_numbriteta_new_conf.shp")

print("Valmis.")

In [68]:
#ERRATA! See jätab servaalad tuvastamata!
#Järgmises toodud skript, mis lõikab terve kaardilehe. Kui sellega kaasneb probleeme, mida selle raku skriptiga ei esinenud, nende lahendusi ei ole käsitletud.

#5. Mitmemodaalse mudeliga tuvastus Gemini 3 näitel.

#5a. Ettevalmistused: ruutudeks jagamine (tiling).
#Lõhume kaardi 6x6 ruutudeks. Need ruudud koos kaasneva indeksifailiga pakkida zip-failiks.

#Samasse kausta, kus on zip-kaust, panna üks hea näidispilt tuvastatavast nimest.

tif_path = "64593_epk_20T_mosaic_bin_new.tif" #cv2 alati ei toeta täpitähtedega failiteesid.

out_dir = Path("64593_epk_20T_tiles_6x6")
out_dir.mkdir(parents=True, exist_ok=True)

rows = 6
cols = 6
overlap_frac = 0.12   # 12% ülekate

with rasterio.open(tif_path) as src:

    w = src.width
    h = src.height
    transform = src.transform
    crs = src.crs

    tile_w = math.ceil(w / cols)
    tile_h = math.ceil(h / rows)

    step_x = int(tile_w * (1 - overlap_frac))
    step_y = int(tile_h * (1 - overlap_frac))

    tiles_meta = []
    tile_id = 0

    for row in range(rows):
        for col in range(cols):

            x0 = col * step_x
            y0 = row * step_y

            window = Window(x0, y0, tile_w, tile_h)

            tile = src.read(
                1,
                window=window,
                boundless=True,
                fill_value=255
            )

            tile = tile.astype(np.float32)

            min_val = tile.min()
            max_val = tile.max()

            if max_val > min_val:
                tile = (tile - min_val) / (max_val - min_val) * 255
            else:
                tile[:] = 255

            tile = tile.astype(np.uint8)

            tile = cv2.equalizeHist(tile)

            tile_name = f"tile_{tile_id:02d}.png"
            tile_path = out_dir / tile_name

            cv2.imwrite(str(tile_path), tile)

            tiles_meta.append({
                "tile_id": tile_id,
                "file": tile_name,
                "x": int(x0),
                "y": int(y0),
                "width": int(tile.shape[1]),
                "height": int(tile.shape[0])
            })

            tile_id += 1


#ruudu indeks
tiles_index = {
    "image": tif_path,
    "image_width": w,
    "image_height": h,
    "rows": rows,
    "cols": cols,
    "overlap_frac": overlap_frac,
    "crs": str(crs),
    "tiles": tiles_meta
}

with open(out_dir / "tiles_index.json", "w", encoding="utf-8") as f:
    json.dump(tiles_index, f, indent=2)

print(f"Salvestatud {tile_id} ruuduga kaust {out_dir}.")

Salvestatud 36 ruuduga kaust C:\Users\antti\Documents\Magister\Maka_matsu\64593_epk_20T_tiles_6x6.


In [ ]:
#Ruutudeks jagamise skript, mis katab terve ala ära.

#5. Mitmemodaalse mudeliga tuvastus trükikaardi ja Gemini 3 näitel.

#5a. Ettevalmistused: ruutudeks jagamine (tiling).

import rasterio
from rasterio.windows import Window
import cv2
import json
import math
import numpy as np
from pathlib import Path

tif_path = "64593_epk_20T_mosaic_bin_new.tif" #cv2 alati ei toeta täpitähtedega failiteesid.

out_dir = Path("64593_epk20T_tiles_6x6_NEW")
out_dir.mkdir(parents=True, exist_ok=True)

rows = 6
cols = 6
overlap_frac = 0.12

with rasterio.open(tif_path) as src:

    w = src.width
    h = src.height
    transform = src.transform
    crs = src.crs

    tile_w = math.ceil(w / cols)
    tile_h = math.ceil(h / rows)

    step_x = max(1, int(tile_w * (1 - overlap_frac)))
    step_y = max(1, int(tile_h * (1 - overlap_frac)))

    x_positions = list(range(0, w - tile_w + 1, step_x))
    if x_positions[-1] != w - tile_w:
        x_positions.append(w - tile_w)

    y_positions = list(range(0, h - tile_h + 1, step_y))
    if y_positions[-1] != h - tile_h:
        y_positions.append(h - tile_h)

    tiles_meta = []
    tile_id = 0

    for y0 in y_positions:
        for x0 in x_positions:

            window = Window(x0, y0, tile_w, tile_h)

            tile = src.read(1, window=window, boundless=True, fill_value=255)

            tile = tile.astype(np.float32)

            min_val = tile.min()
            max_val = tile.max()

            if max_val > min_val:
                tile = (tile - min_val) / (max_val - min_val) * 255
            else:
                tile[:] = 255

            tile = tile.astype(np.uint8)
            tile = cv2.equalizeHist(tile)

            tile_name = f"tile_{tile_id:02d}.png"
            tile_path = out_dir / tile_name

            cv2.imwrite(str(tile_path), tile)

            tiles_meta.append({
                "tile_id": tile_id,
                "file": tile_name,
                "x": int(x0),
                "y": int(y0),
                "width": int(tile.shape[1]),
                "height": int(tile.shape[0])
            })

            tile_id += 1

print("Valmis.")

In [ ]:
#5b. Keelemudeli tuvastusskript, Gemini 3.
#Kopeerida see Google Colab keskkonda.

# @title Run OCR on map tiles with Gemini Flash

!pip install requests pillow

import os
import base64
import requests
import json
import time
import zipfile
from google.colab import files

# ---------------- CONFIG ----------------

OPENROUTER_API_KEY = "INSERT_KEY" #Siia tuleb isiklik OpenRouter võti.
MODEL_ID = "google/gemini-3-flash-preview" #soovi korral vahetada.

# ----------------------------------------


def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def ocr_tile(tile_path, example_label_path):

    tile_b64 = encode_image(tile_path)
    example_b64 = encode_image(example_label_path)

    payload = {
        "model": MODEL_ID,
        "temperature": 0,
        "messages": [
            {
                "role": "user",
                "content": [

                    {
                        "type": "text",
                        "text": """
This map is in Estonian. Read all horizontal text labels from this historical map tile.

Detect even faint or partially visible labels.

Only extract labels written horizontally.

Return JSON exactly like this:

[
 {"name":"Tartu","x":111,"y":222},
 {"name":"Tallinn","x":333,"y":333}
]
"""
                    },

                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{example_b64}"
                        }
                    },

                    {
                        "type": "text",
                        "text": "Example label style above. Now analyze the map tile below."
                    },

                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{tile_b64}"
                        }
                    }

                ]
            }
        ]
    }

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    r = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json=payload,
        timeout=120
    )

    r.raise_for_status()

    return r.json()["choices"][0]["message"]["content"]


# --------------------------------------------------
# 1. Laadi üles ZIP ja näidisnimi
# --------------------------------------------------

print("Upload TWO files:")
print("1) tiles ZIP")
print("2) example label image")

uploaded = files.upload()


zip_file = None
example_label = None

for f in uploaded.keys():
    if f.lower().endswith(".zip"):
        zip_file = f
    elif f.lower().endswith((".png", ".jpg", ".jpeg")):
        example_label = f

if zip_file is None:
    raise Exception("ZIP file not found")

if example_label is None:
    raise Exception("Example label image not found")


print("ZIP file:", zip_file)
print("Example label:", example_label)


# --------------------------------------------------
# 2. Ruutudeks jagamine
# --------------------------------------------------

os.makedirs("tiles", exist_ok=True)

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("tiles")

print("Tiles extracted.")


# --------------------------------------------------
# 3. Indeksifail
# --------------------------------------------------

index_path = "tiles/tiles_index.json"

if not os.path.exists(index_path):
    raise Exception("tiles_index.json not found inside ZIP")

with open(index_path, "r", encoding="utf-8") as f:
    tile_index = {t["file"]: t for t in json.load(f)["tiles"]}

print("Loaded tile index.")


# --------------------------------------------------
# 4. Tuvastus
# --------------------------------------------------

results = []

for tile in sorted(os.listdir("tiles")):

    if not tile.lower().endswith(".png"):
        continue

    tile_path = os.path.join("tiles", tile)

    print("OCR:", tile)

    try:
        text = ocr_tile(tile_path, example_label)

    except Exception as e:
        print("FAILED:", tile, e)
        text = None

    results.append({
        "tile": tile,
        "text": text
    })

    time.sleep(1.2)  # rate limit


# --------------------------------------------------
# 5. Salvesta
# --------------------------------------------------

out_file = "openrouter_ocr_results_64593_20T_6x6_gemini3.json"

with open(out_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Results saved.")

files.download(out_file)

In [71]:
#5c. Tulemuste puhastamine: formaat korda ja jäetakse ainult koordinaatidega tulemused. Ka arvuliste tulemuste eemaldamine.
#Kui antakse veateade "Viga: tile_XX.png", on soovituslik viga käsitsi ära parandada ja käivitada skript uuesti, sest vea liik on etteaimamatu.

input_file = "openrouter_ocr_results_64593_20T_6x6_gemini3.json"
output_file = "openrouter_cleaned_ocr_result_64593_20T_6x6_gemini3.json"

with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

cleaned = []

for entry in data:

    tile = entry["tile"]
    txt = entry["text"]

    if txt is None:
        continue

    #eemaldada "```json" tulemustest.
    txt = re.sub(r"```json|```", "", txt).strip() #Kui on kindel, et formaat on läbivalt korras, võib selle eemaldada.

    try:
        objs = json.loads(txt)
    except:
        print("Viga:", tile)
        continue

    for obj in objs:

        #x ja y asemel tihtipeale label_x/label_y -> viime need samale kujule.
        if "label_x" in obj and "x" not in obj:
            obj["x"] = obj.pop("label_x")

        if "label_y" in obj and "y" not in obj:
            obj["y"] = obj.pop("label_y")

        #Säilitame ainult korrektsed tulemused.
        if not all(k in obj for k in ("name", "x", "y")):
            print("Eemaldatakse tuvastus, millel koordinaadid puuduvad:", obj)
            continue

        #Kontrollime, et koordinaadid on arvulised.
        try:
            obj["x"] = float(obj["x"])
            obj["y"] = float(obj["y"])
        except:
            print("Koordinaadiviga:", obj)
            continue

        obj["tile"] = tile
        cleaned.append(obj)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(cleaned, f, ensure_ascii=False, indent=2)

print("Salvestatud:", output_file,".")

Salvestatud: C:/Users/antti/Documents/Magister/Makatöö/Uudet/Tulemusfailid/openrouter_cleaned_ocr_result_64593_20T_6x6_gemini3_valid.json .


In [72]:
#Eemaldame arvulised tulemused (kõrgusandmed jt).

#Loe eelmises saadud JSON.
with open("openrouter_cleaned_ocr_result_64593_20T_6x6_gemini3.json", "r", encoding="utf-8") as f:
    data = json.load(f)

#Säilitada ainult tulemused, milles on täht.
def has_letter(text):
    return any(char.isalpha() for char in text)

cleaned_data = [item for item in data if has_letter(item["name"])]

#Kirjuta tulemused uute faili.
with open("openrouter_cleaned_ocr_results_64593_20T_6x6_gemini3_numbriteta.json", "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, ensure_ascii=False, indent=2)

print(f"Puhtarvulised tulemused eemaldatud. Jäänud {len(cleaned_data)} tulemust.")

Puhtnumbrilised tulemused eemaldatud. Jäänud 106 tulemust.


In [80]:
#5d. Pikslikoordinaaditide muutmine ristkoordinaatideks ja tuvastatud punktide faili loomine.

#Selle koha peal tuleb tihtipeale ette süstemaatiline nihe.
#Proovida kõigepealt ilma, aga kui punktid on vale koha peal,
#teha mõned kontrollmõõtmised võttes eelmises sammus loodud punkte koordinaatidega
#ja mõõtes nende pikslikoordinaadid nt Paintis ja paluda tehisarul leida vea suurus.
#Võib ka teha GIS-is visuaalse vaatluse, et mis suunas punkte liigutada.
#Teine alternatiiv on, et punktid on klasterdunud iga ruudu vasakusse ülemnurka.
#Sellel juhul liikuda kohta 5e.

clean_results = "openrouter_cleaned_ocr_results_64593_20T_6x6_gemini3_numbriteta.json"
#Eelmise tulemus.
tiles_index_file = "64593_epk_20T_tiles_6x6/tiles_index.json"
#tiles_index_fail on loodud tehes ruudustikku.
raster_file = "64593_epk_20T_mosaic_bin_new.tif"
#raster_fileks panna algne fail, mille pealt tuvastus tehti.

points_out = "64593_gemini3_ocr_points6x6.shp"

#scale= X.XX Muuta vajadusel.

#Loe failid.
with open(clean_results, "r", encoding="utf-8") as f:
    detections = json.load(f)

with open(tiles_index_file, "r", encoding="utf-8") as f:
    tile_index = json.load(f)

tile_lookup = {t["file"]: t for t in tile_index["tiles"]}

points = []
names = []

with rasterio.open(raster_file) as src:
    transform = src.transform

    for d in detections:
        tile_info = tile_lookup[d["tile"]]

        px = d["x"] + tile_info["x"]  #Sest pikslid on iga ruudu kohta eraldi arvutatud, mitte terve kaardi löikes.
        py = d["y"] + tile_info["y"]

        #süstemaatilise nihke parandus, kui on liitmistehe.
        #px += 35 Vahetada arvud ja pluss/miinus vastavalt tulemustele.
        #py += 35

        #Kui vea puhul tegemist on korrutisega:
        #px = tile_info["x"] + d["x"] * scale
        #py = tile_info["y"] + d["y"] * scale

        mx, my = rasterio.transform.xy(transform, py, px)
        points.append(Point(mx, my))
        names.append(d["name"])

gdf_points = gpd.GeoDataFrame({"name": names}, geometry=points, crs="EPSG: 3301")
gdf_points.to_file(points_out)#, driver="ESRI Shapefile")

print("Salvestatud fail:", points_out)

Salvestatud fail: C:/Users/antti/Documents/Magister/Makatöö/Uudet/Tulemusfailid/64593_gemini3_ocr_points6x6.shp


In [ ]:
#5d.1. Näide, kuidas korduvaid punkte on testandmete abil eemaldatud.

#Eemaldame testiandmete abil korduvad nimed ühe pindobjekti seest.

points_file = "64593_gemini3_ocr_points6x6.shp"
polygons_file = "64593_testandmestik.shp"

output_file = "64593_gemini3_ocr_points6x6_corrected_is.shp"
#Kui on mitu, mis on erinevad, valida suure tähega ja kõige pikem.

#andmete lugemine
points = gpd.read_file(points_file)
polygons = gpd.read_file(polygons_file)

#koordinaatsüsteemi kontroll
polygons = polygons.to_crs(points.crs)

# ruumiline ühendamine
joined = gpd.sjoin(
    points,
    polygons,
    predicate="within",
    how="inner"
)

#pindobjektide tsentroidid
poly_centroids = polygons.centroid

joined["poly_centroid"] = joined["index_right"].map(poly_centroids)

joined["dist"] = joined.geometry.distance(joined["poly_centroid"])

joined["has_upper"] = joined["name"].str.contains(r"[A-ZÄÖÕÜ]")
joined["len"] = joined["name"].str.len()

clean = (
    joined
    .sort_values(
        ["index_right", "has_upper", "len", "dist"],
        ascending=[True, False, False, True]
    )
    .drop_duplicates(subset=["index_right"])
)

#säilitada ainult algsed veerud
clean = clean[points.columns]

clean.to_file(output_file)

print("Salvestatud:", output_file)

In [ ]:
##5e. Andmete klasterdumise lahendamine teise lehe näitel.
#Siin on näidiseks teine leht, sest 64593-s andmed ei klasterdunud.
#Selgitada järgmised väärtused ja käivitada järgmine lõik.

all_x = [obj["x"] for obj in detections]
all_y = [obj["y"] for obj in detections]

min_x = min(all_x)
max_x = max(all_x)
min_y = min(all_y)
max_y = max(all_y)

print("VÄÄRTUSED:")
print("min_x =", min_x)
print("max_x =", max_x)
print("min_y =", min_y)
print("max_y =", max_y)
print("width =", max_x - min_x)
print("height =", max_y - min_y)

In [ ]:
clean_results = "openrouter_cleaned_ocr_results_54654_prg_gemini2_5_6x6_numbriteta.json"
#Eelmise tulemus.
tiles_index_file = "54654_praegune_tiles_6x6/tiles_index.json"
#tiles_index_fail on loodud tehes ruudustikku.
raster_file = "54654_mosaic.tif"
#raster_fileks panna algne fail, mille pealt tuvastus tehti.

points_out = "54654_gemini2_5_ocr_points_6x6.shp"

#scale= X.XX Muuta vajadusel.

#Loe failid.
with open(clean_results, "r", encoding="utf-8") as f:
    detections = json.load(f)

with open(tiles_index_file, "r", encoding="utf-8") as f:
    tile_index = json.load(f)

tile_lookup = {t["file"]: t for t in tile_index["tiles"]}

points = []
names = []

with rasterio.open(raster_file) as src:
    transform = src.transform

    for d in detections:
        tile_info = next(tile for tile in tile_index["tiles"] if tile["file"] == d["tile"])

        tile_width_original = tile_info["width"]
        tile_height_original = tile_info["height"]

        OCR_MIN_X = 10.0 #Need väärtused muuta vastavalt eelmises saadule.
        OCR_MIN_Y = 9.0
        OCR_WIDTH = 985.0
        OCR_HEIGHT = 991.0

        scale_x = tile_info["width"] / OCR_WIDTH
        scale_y = tile_info["height"] / OCR_HEIGHT

        px_tile = (d["x"] - OCR_MIN_X) * scale_x
        py_tile = (d["y"] - OCR_MIN_Y) * scale_y

        px_raster = px_tile + tile_info["x"]
        py_raster = py_tile + tile_info["y"]

        mx, my = rasterio.transform.xy(transform, py_raster, px_raster)
        points.append(Point(mx, my))
        names.append(d["name"])

gdf_points = gpd.GeoDataFrame({"name": names}, geometry=points, crs="EPSG: 3301")
gdf_points.to_file(points_out)#, driver="ESRI Shapefile")

#print(d["tile"], tile_width_original, tile_height_original, ocr_tile_width, ocr_tile_height)

print("Salvestatud fail:", points_out)

In [8]:
#6. Kombineeritud meetod - keelemudel tuvastab nimed, mis jäävad EasyOCR-i ristkülikute sisse.

#6a. Nimede väljaLõikamine bboxide abil.

import geopandas as gpd
import rasterio
from rasterio.transform import rowcol

gdf = gpd.read_file("64593_bbox_mosaic_200.geojson")
#Varasemas faasis loodud bboxid.

indexed_crops = []

pad = 10  # Padding = pisut ruumi teksti ümber, et kaasata nt täpitähtede täpid.

tuvastatud_raster = Path("64593_epk_mv_2026/64593_ymbrus/64593_mosaic_3301.tif")

with rasterio.open(tuvastatud_raster) as src:

    #img = src.read(1)
    #img = (img * 255).astype("uint8")
    #transform = src.transform

    img = src.read(1)

    if img.max() <= 1:
        img = (img * 255).astype("uint8") #Need kaks on uus, sest tulemuseks võivad muidu olla mustad kastid.
    else:
        img = img.astype("uint8")
    transform = src.transform

    for i, geom in enumerate(gdf.geometry):

        minx, miny, maxx, maxy = geom.bounds

        r0, c0 = rowcol(transform, minx, maxy)
        r1, c1 = rowcol(transform, maxx, miny)

        r0 = max(0, r0 - pad)
        c0 = max(0, c0 - pad)
        r1 = min(img.shape[0], r1 + pad)
        c1 = min(img.shape[1], c1 + pad)

        crop = img[r0:r1, c0:c1]

        indexed_crops.append({
            "id": i,
            "crop": crop,
            "geom": geom
        })
        
print("Valmis.")

Valmis.


In [ ]:
#6b. Võre loomine.

import cv2
import numpy as np

def make_grid(images, cols=5, size=300):

    rows = int(np.ceil(len(images)/cols))

    grid = np.ones((rows*size, cols*size), dtype=np.uint8)*255

    for i,img in enumerate(images):

        r = i // cols
        c = i % cols

        img = cv2.resize(img,(size,size))

        grid[
            r*size:(r+1)*size,
            c*size:(c+1)*size
        ] = img

    return grid

In [10]:
#6c. Võre alusel keelemudeli sisendpiltide loomine.
#Pakkida need zip-kausta kokku.

out_dir = Path("64593_prg_grid")
out_dir.mkdir(parents=True, exist_ok=True)

batch_size = 25 #Kui mitu nime ühe pildi peale tuleb.

metadata = [] #Link nime ja asukoha vahel.

print("Vähima ümbritseva ristkülikuga nimi leitud kokku:", len(indexed_crops))

for i in range(0, len(indexed_crops), batch_size):

    batch = indexed_crops[i:i+batch_size]

    id_map = {j: item["id"] for j, item in enumerate(batch)}

    images = [c["crop"] for c in batch]

    sheet = make_grid(images)

    filename = out_dir / f"sheet_{i//batch_size:03d}.png"

    ok = cv2.imwrite(str(filename), sheet)

    if ok:
        print("Salvestatud:", filename)
    else:
        print("Ei õnnestunud salvestada faili:", filename)

    metadata.append({
        "sheet": filename.name,
        "id_map": id_map
    })

with open(out_dir / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Metaandmed salvestatud.")

Vähima ümbritseva ristkülikuga nimi leitud kokku: 82
Salvestatud: C:\Users\antti\Documents\Magister\Maka_matsu\64593_prg_grid\sheet_000.png
Salvestatud: C:\Users\antti\Documents\Magister\Maka_matsu\64593_prg_grid\sheet_001.png
Salvestatud: C:\Users\antti\Documents\Magister\Maka_matsu\64593_prg_grid\sheet_002.png
Salvestatud: C:\Users\antti\Documents\Magister\Maka_matsu\64593_prg_grid\sheet_003.png
Metaandmed salvestatud.


In [ ]:
#6d. Tuvastusskript.

#Kopeerida see Google Colab keskkonda.

# @title Run OCR on map tiles with Gemini Flash

!pip install requests pillow

import os
import base64
import requests
import json
import shutil
import time
import zipfile
from google.colab import files

# ---------------- CONFIG ----------------

OPENROUTER_API_KEY = "INSERT_KEY"
MODEL_ID = "google/gemini-3-flash-preview" #soovi korral vahetada.

# ----------------------------------------


def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def ocr_tile(tile_path):

    tile_b64 = encode_image(tile_path)
    #example_b64 = encode_image(example_label_path)

    payload = {
        "model": MODEL_ID,
        "temperature": 0,
        "messages": [
            {
                "role": "user",
                "content": [

                    {
                        "type": "text",
                        "text": """
This map is in Estonian. Grid contains labels arranged row-wise.

Each cell is indexed left-to-right, top-to-bottom:

0  1  2  3  4
5  6  7  8  9
...

Do NOT merge text from multiple cells.
Each cell must be treated independently.


Example:
If a name is split like:
cell 3: "Garmisch-"
cell 4: "Partenkirchen"

Return:
[
 {"cell": 3, "name": "Garmisch-"},
 {"cell": 4, "name": "Partenkirchen"}
]

Return JSON exactly like this:
[
 {"cell":0,"name":"..."},
 {"cell":1,"name":"..."}
]
"""
                    },

                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{tile_b64}"
                        }
                    }

                ]
            }
        ]
    }

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    r = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json=payload,
        timeout=120
    )

    r.raise_for_status()

    return r.json()["choices"][0]["message"]["content"]


# --------------------------------------------------
# 1️ ZIP kausta üleslaadimine.
# --------------------------------------------------

print("Laaditakse kausta (tiles ZIP):")

uploaded = files.upload()

zip_file = None

for f in uploaded.keys():
    if f.lower().endswith(".zip"):
        zip_file = f

if zip_file is None:
    raise Exception("ZIP kausta ei ole.")

print("ZIP kaust:", zip_file)


# --------------------------------------------------
# 2️ Loe ruudud, ennem seda vahemälu tühjendamine.
# --------------------------------------------------

tiles_dir = "tiles"

# Vanade failide kustutamine mälust (kui tehakse sama tuvastust erinevate andmete peal)
if os.path.exists(tiles_dir):
    shutil.rmtree(tiles_dir)

# Tühja kausta loomine uuesti.
os.makedirs(tiles_dir, exist_ok=True)

# Uute ruutude lugemine.
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(tiles_dir)

print("Ruudud loetud.")

# debug
tile_files = sorted(os.listdir(tiles_dir))
print("Ruute leitud:", len(tile_files), "tk.")
print(tile_files)


# --------------------------------------------------
# 3 OCR ruudud.
# --------------------------------------------------

results = []

for tile in sorted(os.listdir("tiles")):

    if not tile.lower().endswith(".png"):
        continue

    tile_path = os.path.join("tiles", tile)

    print("OCR:", tile)

    try:
        text = ocr_tile(tile_path)

    except Exception as e:
        print("VIGA:", tile, e)
        text = None

    results.append({
        "tile": tile,
        "text": text
    })

    time.sleep(1.2)  # rate limit


# --------------------------------------------------
# 4 Tulemuste salvestamine.
# --------------------------------------------------

out_file = "openrouter_ocr_results_64593_prg_Easybbox_gemini3.json"

with open(out_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Tulemused salvestatud.")

files.download(out_file)

In [11]:
#6e. Kombineeritud meetodi tulemuste puhastamine ja tulemusfaili loomine.
#10-tuhandise kaardi jaoks toodud väljakommenteeritud tsükkel.
#Sest 10T kaardilt eemaldatakse tuvastustest rööpnimede sulud.

import json
import re

input_file = "openrouter_ocr_results_64593_prg_Easybbox_gemini3.json"
#Need on eelmise tulemused.
vlm_output_file = "openrouter_ocr_results_64593_prg_Easybbox_gemini3_CLEAN.json"

with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

cleaned = []

for entry in data:

    tile = entry["tile"]
    txt = entry["text"]

    if txt is None:
        continue

    #eemaldada "```json" tulemustest.
    txt = re.sub(r"```json|```", "", txt).strip()
    
    try:
        objs = json.loads(txt)
    except:
        print("Viga:", tile)
        continue

    for obj in objs:
        obj["tile"] = tile
        cleaned.append(obj) #Kui kaart on 10T, kommenteerida need kolm rida välja ja kasutada järgmist.

    #for obj in objs:

        #name = obj.get("name", "")

        #name = name.replace("(", "").replace(")", "")

        #obj["name"] = name.strip()
        #obj["tile"] = tile

        #cleaned.append(obj)

with open(vlm_output_file, "w", encoding="utf-8") as f:
    json.dump(cleaned, f, ensure_ascii=False, indent=2)

print("Salvestatud:", vlm_output_file,".")

Salvestatud: C:/Users/antti/Documents/Magister/Makatöö/Uudet/Tulemusfailid/openrouter_ocr_results_64593_prg_Easybbox_gemini3_grid_newprompt_CLEAN.json .


In [13]:
#Loome shapefaili.
#Kui tekib veateade Key error, kommenteerida välja need kaks plokki if tile... ja if str(cell)...

points = []
names = []

import json
import geopandas as gpd

#Keelemudeli puhastatud tulemused.
with open(vlm_output_file, "r", encoding="utf-8") as f:
    data = json.load(f)

#Metaandmed. Samas grid-kaustas.
with open("64593_prg_grid/metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

meta_lookup = {m["sheet"]: m["id_map"] for m in metadata}

points = []
names = []

for item in data:

    tile = item["tile"]
    cell = item["cell"]
    name = item["name"]

    #if tile not in meta_lookup:
        #print("Missing sheet:", tile)
        #continue

    id_map = meta_lookup[tile]

    #if str(cell) not in id_map:
        #print("Missing cell:", tile, cell)
        #continue

    original_id = id_map[str(cell)]

    geom = indexed_crops[original_id]["geom"]

    points.append(geom.centroid)
    names.append(name)

gdf = gpd.GeoDataFrame(
    {"name": names},
    geometry=points,
    crs="EPSG:3301"
)

gdf.to_file("64593_easy_bbox_ja_vlm_final.shp")

#Kui eelmine annab veateate "Could not set CRS: EPSG:3301", proovida järgmist:
#epsg_crs = CRS.from_epsg(3301).to_wkt()
#gdf_points = gpd.GeoDataFrame({"name": names}, geometry=points, crs=epsg_crs)
#gdf_points.to_file("64593_easy_bbox_ja_vlm_final.shp", engine="fiona")

print("Valmis.")

Valmis.


In [33]:
#6f. Nimede kokkuliitmine.
#Liita kokku nimed, mille bboxid lõikuvad 5 m puhvri sees ja milles on sidekriips.
#Sest kohati hargtäiendiga nimed tuvastatakse kahe erineva osana.

names_gdf = gpd.read_file("64593_easy_bbox_ja_vlm_final.shp")
#Eelmise tulemus.
bboxes_gdf = gpd.read_file("64593_bbox_mosaic_200.geojson")
#Need samad ristkülikud, millega nimed välja lõigati.

#ühendada ristkülik ja nimi.
joined = gpd.sjoin(names_gdf, bboxes_gdf, predicate="within")

print("Valmis.")

Valmis.


In [34]:
joined["centroid_x"] = joined.geometry.x
joined = joined.sort_values(by="centroid_x")

In [35]:
merged = []
used = set()

joined = joined.sort_values(by="centroid_x")

for i, row1 in joined.iterrows():

    if i in used:
        continue

    name1 = row1["name"]
    geom1 = row1.geometry
    bbox1 = bboxes_gdf.loc[row1["index_right"]].geometry

    merged_flag = False  #liitmise träkker.

    for j, row2 in joined.iterrows():

        if j in used or j == i:
            continue

        bbox2 = bboxes_gdf.loc[row2["index_right"]].geometry

        if (
            #("-" in name1 or "-" in row2["name"]) #Selle tingimusega võis sidekriips olla kus iganes.
            (name1.endswith("-") or row2["name"].startswith("-")) #Selleks, et ei panda kokku nimesid, milles pole sidekriipsu.
            and bbox1.buffer(5).intersects(bbox2.buffer(5)) #Viiemeetriline puhver mõlema tulemuse bboxile. Muuta soovi korral
            and abs(geom1.y - row2.geometry.y) < 50 #Et nimesid püstsuunas kokku ei panda.
            #and geom1.distance(row2.geometry) < 300
        ):

            #liitmise järjekord vasakult paremale
            if row2.geometry.x > geom1.x:

                #ainult kaks nime on lubatud liita.
                n1 = name1.rstrip()
                n2 = row2["name"].lstrip()

                if name1.endswith("-"):
                    merged_name = name1 + n2
                elif row2["name"].startswith("-"):
                    merged_name = n1 + row2["name"]
                else:
                    merged_name = n1 + n2
                merged_geom = bbox1.union(bbox2).centroid

                merged.append({
                    "name": merged_name,
                    "geometry": merged_geom
                })

                used.add(i)
                used.add(j)

                merged_flag = True
                break   #Katkestada pärast kahe liitmist.

    #Kui pole liidetud, söilitatakse algne.
    if not merged_flag:
        merged.append({
            "name": name1,
            "geometry": geom1
        })
        used.add(i)

print("Valmis.")

Valmis.


In [36]:
result = gpd.GeoDataFrame(merged, crs=names_gdf.crs)
result.to_file("64593_prg_easy_bbox_ja_vlm_merged.shp")
print("Valmis.")

Valmis.


In [9]:
#7. Ajaliste kihistute kokkuviimine.
#Siin on kasutatud algset faili, GitHubis fail on juba puhastatud.

#7a. Ettevalmistus.

#Teha omaette fail testala praegustest taludest (punktid, mis jäävad testala sisse).

praegused_talud = gpd.read_file("10303_talud_koopia_trim.shp")
buffer = gpd.read_file("64593_epk_mv_2026/64593_buffer200m.shp")

#Igaks juhuks võib enne lõikumist kontrollida koordinaatsüsteemi. Peavad olema sama (EPSG: 3301).
#print(praegused_talud.crs)
#print(buffer.crs)

talud_64593_200m = gpd.sjoin(
    praegused_talud,
    buffer,
    predicate="intersects"
)

talud_64593_200m.to_file("talud_64593_200m_Python.shp")

C:\Users\antti\AppData\Local\Temp\ipykernel_6216\346117073.py:16: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  talud_64593_200m.to_file("C:/Users/antti/Documents/Magister/Makatöö/Uudet/Tulemusfailid/talud_64593_200m_Python.shp")
C:\Users\antti\micromamba\envs\geo312\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'index_right' to 'index_righ'
  ogr_write(


In [10]:
display(talud_64593_200m.head(5))

,id,nimeobjekt,ametlik_ni,olek,nimeobje_1,liigi_kood,esindusaad,nimi_trim,geometry,index_right,nimi
5494,NaN,100285251,reinumäe,10303,6555287.19,691770,2017-11-06Z,Reinumäe,POINT (691770.11 6555287.19),0,64593_extent
6438,NaN,100284911,sorgi,10303,6555354.530000001,690765,2017-11-06Z,Sorgi,POINT (690764.93 6555354.53),0,64593_extent
6439,NaN,100284912,kriivani,10303,6555329.42,690776,2024-04-13Z,Kriivani,POINT (690776.43 6555329.42),0,64593_extent
6463,NaN,100284922,kaljumäe,10303,6559218.630000001,690076,2024-04-13Z,Kaljumäe,POINT (690075.97 6559218.63),0,64593_extent
6465,NaN,100284924,väike-koldamäe,10303,6559242.4,692182,2024-04-13Z,Väike-koldamäe,POINT (692181.97 6559242.4),0,64593_extent


In [ ]:
#Maa- ja Ruumiameti andmetabel sisaldab veerge, mis ei anna tuvastuse jaoks olulist teavet. Soovi korral võib eemaldada ka mõned teised veerud.
talud_64593_200m_clean = talud_64593_200m.drop(columns=['olek', 'liigi_kood', 'esindusaad', 'index_right', 'nimi'])

In [ ]:
display(talud_64593_200m_clean.head(5))

In [ ]:
#Soovi korral kirjutada faili.
talud_64593_200m_clean.to_file("talud_64593_200m_clean.shp")

In [ ]:
#7b.Kombineeritud meetodi alusel saadud erinevate aegade tulemuste kokkuviiimine (referentsiks praegused nimed).

#Alustame praegustest nimedest ja kehtivast kaardist, aga vaja on ainult tuvastustulemuste kihti muuta.
#See on ühe lehe näitel, pärast on näidis ka mitmele lehele korraga.
#Siin otsime puhvri sees teisenduskaugusega kandidaadid ja valime välja väikseima teisenduskaugusega objekti.
#Kui selliseid leidub mitu, paneme parimaks kandidaadiks väikseima vahemaaga (meetrites) kandidaadi.
#Samal uuritakse, kas kandidaat on "õige" nime alamsõne. Siin salvestatakse, mis on alamsõneks.

def normalize(text):
    return text.lower()

talud_64593_200m_clean = gpd.read_file("talud_64593_200m_clean_Python.shp") #Seda rida pole vaja, kui jooksutada skript järjest algusest peale.
tuvastus = gpd.read_file("64593_prg_easy_bbox_ja_vlm_merged.shp") #Kombineeritud meetodi tulemused.

#Kui koordinaatidega on kahtlusi, kontrollida:
#print(talud_64593_200m_clean.crs)
#print(tuvastus.crs)

#Ja vajadusel ümbrprojitseerida.
#talud_64593_200m = talud_64593_200m_clean.to_crs(epsg=3301)
#tuvastus = tuvastus.to_crs(talud_64593_200m_clean.crs)

output_rows = []

for idx, row in talud_64593_200m_clean.iterrows():
    prg_talunimi = row["ametlik_ni"]
    geom = row.geometry

    #Loo 250 m puhver.
    buffer_talud = geom.buffer(250) #laius meetrites. Tähtis, et koordinaatsüsteemiks on Eesti ristkoordinaatsüsteem.

    #Leia kandidaadid, mis lõikuvad tuvastustulemuste puhvriga.
    candidates = tuvastus[tuvastus.geometry.intersects(buffer_talud)]

    out_row = row.copy()

    out_row["year"] = 2026
    out_row["area_id"] = 64593
    out_row["reference"] = prg_talunimi

    #Tsükliga käiakse üle kandidaadid.
    best = None
    best_lev = None
    best_dist = None

    for i, (_, cand) in enumerate(candidates.iterrows(), start=1):
        text = cand["name"] #See veerg, milles kombineeritud meetodi tulemustes on tuvastustulemus.

        lev = distance(normalize(prg_talunimi), normalize(text)) #Normaliseeritakse väiketähtedeks, sest muidu "metsa" ja "Metsa" kaugus on 1.
        #Kui emb-kumb võib olla None, kasutada seda väljakommenteeritud kohta ja kommenteerida eelmine rida välja.
        #if text is None or prg_talunimi is None:
            #continue
        #lev = distance(normalize(str(prg_talunimi)), normalize(str(text)))
        dist_m = geom.distance(cand.geometry)

        out_row[f"Candidate{i}"] = text
        out_row[f"Cand{i}Lev"] = lev
        out_row[f"Cand{i}Dist"] = dist_m

        #substr = normalize(prg_talunimi) in normalize(text) or normalize(text) in normalize(prg_talunimi) #Paneme kirja, kui kandidaat on nime alamsõne.
        name1 = normalize(prg_talunimi)
        name2 = normalize(text)

        if name1 in name2:
            substr = name1
        elif name2 in name1:
            substr = name2
        else:
            substr = None
            
        out_row[f"Cand{i}Substr"] = substr


        if best is None:
            best = text
            best_lev = lev
            best_dist = dist_m
        else:
            if lev < best_lev:
                best = text
                best_lev = lev
                best_dist = dist_m
            elif lev == best_lev and dist_m < best_dist:
                best = text
                best_dist = dist_m

    out_row["best_cand"] = best
    out_row["best_lev"] = best_lev
    out_row["best_dist_m"] = best_dist           

    output_rows.append(out_row)

result = gpd.GeoDataFrame(output_rows, crs=talud_64593_200m_clean.crs)

#Veergude järjekord -> parim kandidaat enne teisi kandidaate.
base_cols = list(talud_64593_200m_clean.columns)

cand_cols = sorted([c for c in result.columns if c.startswith("Candidate")])
lev_cols = sorted([c for c in result.columns if "Lev" in c and c.startswith("Cand")])
dist_cols = sorted([c for c in result.columns if "Dist" in c and c.startswith("Cand")])
substr_cols = sorted([c for c in result.columns if c.startswith("Cand") and c.endswith("Substr")])

best_cols = ["best_cand", "best_lev", "best_dist_m"]

new_order = base_cols + best_cols + cand_cols + lev_cols + dist_cols + substr_cols

result = result[new_order]

#print(result.columns) #Kui tekib veatede, et sisu ei sobi veergusse, sellega võib uurida põhjust.

result.to_file("64593_ajaline_kokkuviimine_250.gpkg", layer="results", driver="GPKG")

print("Valmis.")

In [ ]:
#7b.1 Näidis mitme kihiga. Praegused nimed ja kehtiv põhikaart.

#Otsime 250 m puhvri sees teisenduskaugusega kandidaadid ja valime välja väikseima teisenduskaugusega objekti.
#Kui selliseid leidub mitu, paneme parimaks kandidaadiks väikseima vahemaaga (meetrites) kandidaadi.
#Samal uuritakse, kas kandidaat on "õige" nime alamsõne. Siis salvestatakse, mis on alamsõneks.

def normalize(text): #Siin on toodud sama funktsioon, mis eel, sest tingimata sisu ei jooksutada järjest.
    return text.lower()

talud_64593 = gpd.read_file("talud_64593_200m_clean_Python.shp")
tuvastus_64593 = gpd.read_file("64593_prg_easy_bbox_ja_vlm_merged.shp")

talud_63644 = gpd.read_file("talud_63644_200m_clean_Python.shp")
tuvastus_63644 = gpd.read_file("63644_prg_easy_bbox_ja_vlm_merged.shp")

talud_62471 = gpd.read_file("talud_62471_200m_clean_Python.shp")
tuvastus_62471 = gpd.read_file("62471_prg_mosaic_easy_bbox_ja_vlm_merged.shp")

talud_54654 = gpd.read_file("talud_54654_200m_clean_Python.shp")
tuvastus_54654 = gpd.read_file("54654_prg_easy_bbox_ja_vlm_merged.shp")

talud_54082 = gpd.read_file("talud_54082_200m_clean_Python.shp")
tuvastus_54082 = gpd.read_file("54082_prg_easy_bbox_ja_vlm_merged.shp")

#Kui koordinaatidega on kahtlusi, kontrollida:
#print(talud_64593_200m_clean.crs)
#print(tuvastus.crs)

#Ja vajadusel ümbrprojitseerida.
#talud_64593_200m = talud_64593_200m_clean.to_crs(epsg=3301)
#tuvastus = tuvastus.to_crs(talud_64593_200m_clean.crs)

aasta = 2026
ala_id = 64593
MAX_CAND = 5

def process_area(talud_gdf, tuvastus_gdf, aasta, ala_id, MAX_CAND=5):

    output_rows = []

    for idx, row in talud_gdf.iterrows():
        prg_talunimi = row["ametlik_ni"]
        geom = row.geometry

        buffer_talud = geom.buffer(250)
        candidates = tuvastus_gdf[tuvastus_gdf.geometry.intersects(buffer_talud)].copy()

        out_row = row.copy()
        out_row["aasta"] = aasta
        out_row["ala_id"] = ala_id

        candidates["dist_tmp"] = candidates.geometry.distance(geom)
        candidates = candidates.sort_values("dist_tmp")

        best = None
        best_lev = float("nan")
        best_dist = float("nan")

        for i, (_, cand) in enumerate(candidates.iterrows(), start=1):
            if i > MAX_CAND:
                break

            text = cand["name"]

            lev = distance(normalize(prg_talunimi), normalize(text))        
            dist_m = geom.distance(cand.geometry)

            out_row[f"Candidate{i}"] = text
            out_row[f"Cand{i}Lev"] = lev
            out_row[f"Cand{i}Dist"] = dist_m

            name1 = normalize(prg_talunimi)
            name2 = normalize(text)

            if name1 in name2:
                substr = name1
            elif name2 in name1:
                substr = name2
            else:
                substr = None

            out_row[f"Cand{i}Substr"] = substr

            if best is None:
                best = text
                best_lev = lev
                best_dist = dist_m
            else:
                if lev < best_lev:
                    best = text
                    best_lev = lev
                    best_dist = dist_m
                elif lev == best_lev and dist_m < best_dist:
                    best = text
                    best_dist = dist_m

        out_row["best_cand"] = best
        out_row["best_lev"] = best_lev
        out_row["best_dist_m"] = best_dist

        output_rows.append(out_row)

    return gpd.GeoDataFrame(output_rows, crs=talud_gdf.crs)

df_64593 = process_area(talud_64593, tuvastus_64593, 2026, 64593)
df_63644 = process_area(talud_63644, tuvastus_63644, 2026, 63644)
df_62471 = process_area(talud_62471, tuvastus_62471, 2026, 62471)
df_54654 = process_area(talud_54654, tuvastus_54654, 2026, 54654)
df_54082 = process_area(talud_54082, tuvastus_54082, 2026, 54082)

result = pd.concat([df_64593, df_63644, df_62471, df_54654, df_54082], ignore_index=True)

#Veergude järjekord -> parim kandidaat enne teisi kandidaate.
best_cols = ["best_cand", "best_lev", "best_dist_m"]

meta_cols = ["aasta", "ala_id"]

cand_cols = sorted([c for c in result.columns if c.startswith("Candidate")])
lev_cols = sorted([c for c in result.columns if "Lev" in c and c.startswith("Cand")])
dist_cols = sorted([c for c in result.columns if "Dist" in c and c.startswith("Cand")])
substr_cols = sorted([c for c in result.columns if c.startswith("Cand") and c.endswith("Substr")])

best_cols = ["best_cand", "best_lev", "best_dist_m"]

other_cols = [c for c in result.columns
              if c not in meta_cols + best_cols + cand_cols + lev_cols + dist_cols + substr_cols]

new_order = other_cols + meta_cols + best_cols + cand_cols + lev_cols + dist_cols + substr_cols

result = result[new_order]

#print(result.columns) #Kui tekib veatede, et sisu ei sobi veergusse, sellega võib uurida põhjust.

for col in result.columns:
    if "Lev" in col or "Dist" in col:
        result[col] = pd.to_numeric(result[col], errors="coerce")

result.to_file("k6ik_v6rdlus_kehtivaga_250.gpkg", layer="results", driver="GPKG")

print("Valmis.")

In [ ]:
#7b.2 Näidis eelmise ajalise kihiga.

#Otsime 250 m puhvri sees teisenduskaugusega kandidaadid ja valime välja väikseima teisenduskaugusega objekti.
#Kui selliseid leidub mitu, paneme parimaks kandidaadiks väikseima vahemaaga (meetrites) kandidaadi.
#Samal uuritakse, kas kandidaat on "õige" nime alamsõne. Siis salvestatakse, mis on alamsõneks.

def normalize(text):
    return text.lower()

#talud_64593_200m_clean = gpd.read_file("C:/Users/antti/Documents/Magister/Makatöö/Uudet/Tulemusfailid/talud_64593_200m_clean_Python.shp") #Seda rida pole vaja, kui jooksutada skript järjest algusest peale.
#tuvastus = gpd.read_file("C:/Users/antti/Documents/Magister/Makatöö/Uudet/Tulemusfailid/64593_prg_easy_bbox_ja_vlm_merged.shp") #Kombineeritud meetodi tulemused.

talud_64593 = gpd.read_file("Ctalud_64593_200m_clean_Python.shp")
tuvastus_64593_eelm = gpd.read_file("64593_20T_easy_bbox_ja_vlm_merged.shp") #Ehk siin võrreldakse praeguseid nimesid eelmise kihiga.

talud_63644 = gpd.read_file("talud_63644_200m_clean_Python.shp")
tuvastus_63644_eelm = gpd.read_file("63644_epk10T_easy_bbox_ja_vlm_merged_newhyphen.shp")

talud_62471 = gpd.read_file("talud_62471_200m_clean_Python.shp")
tuvastus_62471_eelm = gpd.read_file("62471_20T_easy_bbox_ja_vlm_merged.shp")

talud_54654 = gpd.read_file("talud_54654_200m_clean_Python.shp")
tuvastus_54654_eelm = gpd.read_file("54654_20T_easy_bbox_ja_vlm_merged.shp")

talud_54082 = gpd.read_file("talud_54082_200m_clean_Python.shp")
tuvastus_54082_eelm = gpd.read_file("54082_10T_easy_bbox_ja_vlm_merged.shp")

#Kui koordinaatidega on kahtlusi, kontrollida:
#print(talud_64593_200m_clean.crs)
#print(tuvastus.crs)

#Ja vajadusel ümbrprojitseerida.
#talud_64593_200m = talud_64593_200m_clean.to_crs(epsg=3301)
#tuvastus = tuvastus.to_crs(talud_64593_200m_clean.crs)

MAX_CAND = 5

def process_area(talud_gdf, tuvastus_gdf, aasta, ala_id, MAX_CAND=5):

    output_rows = []

    for idx, row in talud_gdf.iterrows():
        prg_talunimi = row["ametlik_ni"]
        geom = row.geometry

        buffer_talud = geom.buffer(250)
        candidates = tuvastus_gdf[tuvastus_gdf.geometry.intersects(buffer_talud)].copy()

        out_row = row.copy()
        out_row["aasta"] = aasta
        out_row["ala_id"] = ala_id

        candidates["dist_tmp"] = candidates.geometry.distance(geom)
        candidates = candidates.sort_values("dist_tmp")

        best = None
        best_lev = float("nan")
        best_dist = float("nan")

        for i, (_, cand) in enumerate(candidates.iterrows(), start=1):
            if i > MAX_CAND:
                break

            text = cand["name"]

            lev = distance(normalize(prg_talunimi), normalize(text))              
            dist_m = geom.distance(cand.geometry)

            out_row[f"Candidate{i}"] = text
            out_row[f"Cand{i}Lev"] = lev
            out_row[f"Cand{i}Dist"] = dist_m

            name1 = normalize(prg_talunimi)
            name2 = normalize(text)

            if name1 in name2:
                substr = name1
            elif name2 in name1:
                substr = name2
            else:
                substr = None

            out_row[f"Cand{i}Substr"] = substr

            if best is None:
                best = text
                best_lev = lev
                best_dist = dist_m
            else:
                if lev < best_lev:
                    best = text
                    best_lev = lev
                    best_dist = dist_m
                elif lev == best_lev and dist_m < best_dist:
                    best = text
                    best_dist = dist_m

        out_row["best_cand"] = best
        out_row["best_lev"] = best_lev
        out_row["best_dist_m"] = best_dist

        output_rows.append(out_row)

    return gpd.GeoDataFrame(output_rows, crs=talud_gdf.crs)

areas = [
    {"talud": talud_64593, "tuvastus": tuvastus_64593_eelm, "aasta": 2016, "ala_id": 64593}, #Selle koha peal tuöeb sättida aastad.
    {"talud": talud_63644, "tuvastus": tuvastus_63644_eelm, "aasta": 2004, "ala_id": 63644},
    {"talud": talud_62471, "tuvastus": tuvastus_62471_eelm, "aasta": 2004, "ala_id": 62471},
    {"talud": talud_54654, "tuvastus": tuvastus_54654_eelm, "aasta": 2006, "ala_id": 54654},
    {"talud": talud_54082, "tuvastus": tuvastus_54082_eelm, "aasta": 2003, "ala_id": 54082},
]

dfs = []

for area in areas:
    df = process_area(
        area["talud"],
        area["tuvastus"],
        area["aasta"],
        area["ala_id"],
        MAX_CAND
    )
    dfs.append(df)

result = pd.concat(dfs, ignore_index=True)

#Veergude järjekord -> parim kandidaat enne teisi kandidaate.
best_cols = ["best_cand", "best_lev", "best_dist_m"]

meta_cols = ["aasta", "ala_id"]

cand_cols = sorted([c for c in result.columns if c.startswith("Candidate")])
lev_cols = sorted([c for c in result.columns if "Lev" in c and c.startswith("Cand")])
dist_cols = sorted([c for c in result.columns if "Dist" in c and c.startswith("Cand")])
substr_cols = sorted([c for c in result.columns if c.startswith("Cand") and c.endswith("Substr")])

best_cols = ["best_cand", "best_lev", "best_dist_m"]

other_cols = [c for c in result.columns
              if c not in meta_cols + best_cols + cand_cols + lev_cols + dist_cols + substr_cols]

new_order = other_cols + meta_cols + best_cols + cand_cols + lev_cols + dist_cols + substr_cols

result = result[new_order]

#print(result.columns)

for col in result.columns:
    if "Lev" in col or "Dist" in col:
        result[col] = pd.to_numeric(result[col], errors="coerce") #coarce= "If a value can be converted to a number, convert it."

result.to_file("k6ik_v6rdlus_eelmisega_250.gpkg", layer="results", driver="GPKG")

print("Valmis.")

In [ ]:
#7c. Teisenduskauguse alusel kahe tuvastustulemuse kokkuviimine. Näidiseks 0-kaugus, et saada need, mis pole muutunud.
#Eelmistes võrreldi praeguste talunimedega, siin sisendiks on eelmise kihi tingimusele vastavad nimed.
#Toodud ka võimalus kaugust muuta.

def normalize(text):
    return text.lower()

praegune = gpd.read_file("k6ik_v6rdlus_kehtivaga_250.gpkg", layer="results") #Fail, mis on saadud kohas 7b.1.

tuvastus_64593_eelm = gpd.read_file("64593_20T_easy_bbox_ja_vlm_merged.shp")

tuvastus_63644_eelm = gpd.read_file("63644_epk10T_easy_bbox_ja_vlm_merged_newhyphen.shp")

tuvastus_62471_eelm = gpd.read_file("62471_20T_easy_bbox_ja_vlm_merged.shp")

tuvastus_54654_eelm = gpd.read_file("54654_20T_easy_bbox_ja_vlm_merged.shp")

tuvastus_54082_eelm = gpd.read_file("54082_10T_easy_bbox_ja_vlm_merged.shp")


#Ainult need, mille teisenduskaugus on 0.
praegune_matched = praegune[praegune["best_lev"] == 0].copy()
#kui on soov kaugust muuta:
#praegune_matched = praegune[praegune["best_lev"].isin([0, 1])].copy()
#Sättida eelmises numbrid vastavalt oma eesmärgile. Nt Isin 0,1,2,3 võtab kaugused 0 kuni 3, kaasa arvatud.
#Ja kommenteerida esimene praegune_matched välja.


# Optional: remove rows with missing names
#praegune_matched = praegune_matched[praegune_matched["best_cand"].notna()].copy()

def process_previous_layer(current_gdf, previous_gdf, aasta, ala_id, MAX_CAND=5):

    output_rows = []

    for idx, row in current_gdf.iterrows():

        prg_talunimi = row["best_cand"]
        geom = row.geometry

        buffer_talud = geom.buffer(250)

        candidates = previous_gdf[
            previous_gdf.geometry.intersects(buffer_talud)
        ].copy()

        out_row = {
            "ametlik_ni": row["ametlik_ni"],
            "best_cand": row["best_cand"],
            "best_dist_m": row["best_dist_m"],
            "aasta": aasta,
            "ala_id": ala_id,
            "geometry": row.geometry
        }

        candidates["dist_tmp"] = candidates.geometry.distance(geom)
        candidates = candidates.sort_values("dist_tmp")

        best = None
        best_lev = float("nan")
        best_dist = float("nan")

        for i, (_, cand) in enumerate(candidates.iterrows(), start=1):

            if i > MAX_CAND:
                break

            text = cand["name"]

            lev = distance(normalize(prg_talunimi), normalize(text))

            #if lev >1:
                    #continue
                
            dist_m = geom.distance(cand.geometry)

            out_row[f"PrevCandidate{i}"] = text
            out_row[f"PrevCand{i}Lev"] = lev
            out_row[f"PrevCand{i}Dist"] = dist_m

            name1 = normalize(prg_talunimi)
            name2 = normalize(text)

            if name1 in name2:
                substr = name1
            elif name2 in name1:
                substr = name2
            else:
                substr = None

            out_row[f"PrevCand{i}Substr"] = substr

            if best is None:
                best = text
                best_lev = lev
                best_dist = dist_m
            else:
                if lev < best_lev:
                    best = text
                    best_lev = lev
                    best_dist = dist_m
                elif lev == best_lev and dist_m < best_dist:
                    best = text
                    best_dist = dist_m

        out_row["best_prev"] = best
        out_row["best_prev_lev"] = best_lev
        out_row["best_prev_dist_m"] = best_dist

        output_rows.append(out_row)

    return gpd.GeoDataFrame(output_rows, crs=current_gdf.crs)

cur_64593 = praegune_matched[praegune_matched["ala_id"] == 64593]
cur_63644 = praegune_matched[praegune_matched["ala_id"] == 63644]
cur_62471 = praegune_matched[praegune_matched["ala_id"] == 62471]
cur_54654 = praegune_matched[praegune_matched["ala_id"] == 54654]
cur_54082 = praegune_matched[praegune_matched["ala_id"] == 54082]

df_64593_prev = process_previous_layer(cur_64593, tuvastus_64593_eelm, 2016, 64593)
df_63644_prev = process_previous_layer(cur_63644, tuvastus_63644_eelm, 2004, 63644)
df_62471_prev = process_previous_layer(cur_62471, tuvastus_62471_eelm, 2004, 62471)
df_54654_prev = process_previous_layer(cur_54654, tuvastus_54654_eelm, 2006, 54654)
df_54082_prev = process_previous_layer(cur_54082, tuvastus_54082_eelm, 2003, 54082)

result_prev = pd.concat([
    df_64593_prev,
    df_63644_prev,
    df_62471_prev,
    df_54654_prev,
    df_54082_prev
], ignore_index=True)

base_cols = ["ametlik_ni", "best_cand", "best_dist_m", "aasta", "ala_id", "geometry"]

prev_cand_cols = sorted([c for c in result_prev.columns if c.startswith("PrevCandidate")])
prev_lev_cols = sorted([c for c in result_prev.columns if c.startswith("PrevCand") and "Lev" in c])
prev_dist_cols = sorted([c for c in result_prev.columns if c.startswith("PrevCand") and "Dist" in c])
prev_substr_cols = sorted([c for c in result_prev.columns if c.startswith("PrevCand") and c.endswith("Substr")])

prev_best_cols = ["best_prev", "best_prev_lev", "best_prev_dist_m"]

new_order = base_cols + prev_best_cols + prev_cand_cols + prev_lev_cols + prev_dist_cols + prev_substr_cols

result_prev = result_prev[new_order]

result_prev.to_file(
    "kombi_lev0_eelmisega.gpkg", #Järgmise (vanema) kihiga toimetades saab see niisiis sisendfailiks.
    layer="results",
    driver="GPKG"
)

print("Valmis.")